In [ ]:
"""
RELOAD FULL DATASETS
===================

Use this cell to reload train.csv and test.csv in their entirety
Then re-run preprocessing
"""

import pandas as pd
import numpy as np

print("=" * 80)
print("RELOADING FULL DATASETS")
print("=" * 80)

# Load FULL train data
print("\n[STEP 1] Loading FULL train.csv...")
train_raw = pd.read_csv('train.csv')
print(f"✓ Loaded {len(train_raw)} rows × {len(train_raw.columns)} columns")
print(f"  Churn distribution: {train_raw['Churn'].value_counts().to_dict()}")

# Load FULL test data
print("\n[STEP 2] Loading FULL test.csv...")
test_raw = pd.read_csv('test.csv')
print(f"✓ Loaded {len(test_raw)} rows × {len(test_raw.columns)} columns")
print(f"  ID range: {test_raw['id'].min()} to {test_raw['id'].max()}")

print("\n[STEP 3] Running preprocessing pipeline...")
print("  (Make sure preprocessing_pipeline function is loaded in previous cell)\n")

# Run preprocessing
X_train, y_train, encoders = preprocess_pipeline(train_raw, fit_encoders=True)
X_test, _, _ = preprocess_pipeline(test_raw, fit_encoders=False, encoders_dict=encoders)

print("\n" + "=" * 80)
print("DATASETS READY")
print("=" * 80)
print(f"\n✓ X_train: {X_train.shape}")
print(f"✓ y_train: {y_train.shape}")
print(f"✓ X_test: {X_test.shape}")
print(f"✓ test_raw: {test_raw.shape}")

print(f"\n[STEP 4] Creating train/val split...")
from sklearn.model_selection import train_test_split

X_train_tuned, X_val, y_train_tuned, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"✓ X_train_tuned: {X_train_tuned.shape}")
print(f"✓ X_val: {X_val.shape}")
print(f"✓ y_train_tuned: {y_train_tuned.shape}")
print(f"✓ y_val: {y_val.shape}")

print("\n" + "=" * 80)
print("✓ READY FOR MODELING!")
print("=" * 80)
print(f"\nNow run XGBoost cell to train on FULL dataset!")

RELOADING FULL DATASETS

[STEP 1] Loading FULL train.csv...
✓ Loaded 594194 rows × 21 columns
  Churn distribution: {'No': 460377, 'Yes': 133817}

[STEP 2] Loading FULL test.csv...
✓ Loaded 254655 rows × 20 columns
  ID range: 594194 to 848848

[STEP 3] Running preprocessing pipeline...
  (Make sure preprocessing_pipeline function is loaded in previous cell)



NameError: name 'preprocess_pipeline' is not defined

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# LOAD DATA - UPDATE THIS PATH IF NEEDED
# ============================================================================
train = pd.read_csv('train.csv')
print("=" * 80)
print("LOADING DATA...")
print("=" * 80)
print(f"✓ Loaded {len(train)} rows × {len(train.columns)} columns\n")

# ============================================================================
# 1. BASIC INFO
# ============================================================================
print("=" * 80)
print("1. DATASET OVERVIEW")
print("=" * 80)
print(f"Shape: {train.shape}")
print(f"\nData types:\n{train.dtypes}\n")
print(f"Missing values:\n{train.isnull().sum()}\n")
print(f"Target class distribution:\n{train['Churn'].value_counts()}")
print(f"Churn rate: {(train['Churn'] == 'Yes').sum() / len(train) * 100:.2f}%\n")

# ============================================================================
# 2. NUMERIC FEATURES
# ============================================================================
print("=" * 80)
print("2. NUMERIC FEATURES STATISTICS")
print("=" * 80)
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'id']
print(train[numeric_cols].describe())
print()

# ============================================================================
# 3. CATEGORICAL FEATURES
# ============================================================================
print("=" * 80)
print("3. CATEGORICAL FEATURES")
print("=" * 80)
categorical_cols = train.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'Churn']

for col in categorical_cols:
    print(f"\n{col}:")
    print(train[col].value_counts().to_string())

# ============================================================================
# 4. CREATE VISUALIZATIONS
# ============================================================================
print("\n" + "=" * 80)
print("CREATING VISUALIZATIONS...")
print("=" * 80)

numeric_for_viz = ['tenure', 'MonthlyCharges', 'TotalCharges']

# 4a. Churn distribution + key features
fig = plt.figure(figsize=(16, 12))

ax1 = plt.subplot(3, 3, 1)
churn_counts = train['Churn'].value_counts()
ax1.bar(churn_counts.index, churn_counts.values, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Churn Distribution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    ax1.text(i, v + 200, str(v), ha='center', fontweight='bold')

# Numeric features distributions
for idx, col in enumerate(numeric_for_viz):
    ax = plt.subplot(3, 3, idx + 2)
    data_clean = train[col].dropna()
    ax.hist(data_clean, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax.set_title(f'{col} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

# Top categorical features vs Churn
top_categorical = ['InternetService', 'Contract', 'PaymentMethod', 'gender']

for idx, col in enumerate(top_categorical):
    ax = plt.subplot(3, 3, idx + 5)
    churn_by_cat = pd.crosstab(train[col], train['Churn'], normalize='index') * 100
    churn_by_cat['Yes'].plot(kind='barh', ax=ax, color='#e74c3c')
    ax.set_title(f'Churn Rate by {col}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Churn Rate (%)')
    for i, v in enumerate(ax.patches):
        ax.text(v.get_width() + 1, v.get_y() + v.get_height()/2,
                f'{v.get_width():.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('01_churn_overview.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 01_churn_overview.png")
plt.close()

# ============================================================================
# 5. NUMERIC FEATURES vs CHURN
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, col in enumerate(numeric_for_viz):
    ax = axes[idx]
    data_churn = train[train['Churn'] == 'Yes'][col].dropna()
    data_no_churn = train[train['Churn'] == 'No'][col].dropna()

    ax.hist(data_no_churn, bins=40, alpha=0.6, label='No Churn', color='#2ecc71')
    ax.hist(data_churn, bins=40, alpha=0.6, label='Churn', color='#e74c3c')
    ax.set_title(f'{col} by Churn Status', fontsize=12, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.savefig('02_numeric_vs_churn.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 02_numeric_vs_churn.png")
plt.close()

# ============================================================================
# 6. FEATURE IMPORTANCE (via correlation & chi-square)
# ============================================================================
print("\n" + "=" * 80)
print("4. FEATURE CORRELATION WITH CHURN")
print("=" * 80)

# Numeric correlation
churn_binary = (train['Churn'] == 'Yes').astype(int)
correlations = {}
for col in numeric_for_viz:
    corr, pval = pointbiserialr(churn_binary, train[col].fillna(train[col].mean()))
    correlations[col] = abs(corr)
    print(f"{col}: {corr:.4f}")

# Categorical association (Chi-square)
print("\nCategorical features (Chi-square p-values):")
chi_square_results = {}
for col in categorical_cols:
    contingency = pd.crosstab(train[col], train['Churn'])
    chi2, pval, dof, expected = chi2_contingency(contingency)
    chi_square_results[col] = pval
    print(f"{col}: p-value = {pval:.2e} {'***' if pval < 0.001 else ''}")

# ============================================================================
# 7. ALL CATEGORICAL FEATURES CHURN RATES
# ============================================================================
fig = plt.figure(figsize=(18, 14))

for idx, col in enumerate(categorical_cols):
    ax = plt.subplot(4, 4, idx + 1)
    churn_rate = pd.crosstab(train[col], train['Churn'], normalize='index') * 100

    if 'Yes' in churn_rate.columns:
        churn_rate['Yes'].sort_values(ascending=False).plot(
            kind='barh', ax=ax, color='#e74c3c', alpha=0.8
        )
    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Churn Rate (%)')
    ax.set_xlim(0, 100)

    # Add value labels
    for i, v in enumerate(ax.patches):
        if v.get_width() > 0:
            ax.text(v.get_width() - 5, v.get_y() + v.get_height()/2,
                    f'{v.get_width():.0f}%', va='center', ha='right',
                    color='white', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('03_all_categorical_churn_rates.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 03_all_categorical_churn_rates.png")
plt.close()

# ============================================================================
# 8. FEATURE INTERACTIONS
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Contract + InternetService
ax = axes[0, 0]
interaction = pd.crosstab([train['Contract'], train['InternetService']],
                          train['Churn'], normalize='index') * 100
interaction['Yes'].unstack().plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c', '#f39c12'])
ax.set_title('Churn Rate: Contract × InternetService', fontsize=12, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.legend(title='Internet Service')
ax.tick_params(axis='x', rotation=45)

# Contract + tenure (binned)
ax = axes[0, 1]
train['tenure_bin'] = pd.cut(train['tenure'], bins=[0, 12, 24, 36, 72],
                             labels=['0-1yr', '1-2yr', '2-3yr', '3yr+'])
interaction = pd.crosstab([train['Contract'], train['tenure_bin']],
                          train['Churn'], normalize='index') * 100
interaction['Yes'].unstack().plot(kind='bar', ax=ax, color=['#9b59b6', '#1abc9c', '#e67e22', '#c0392b'])
ax.set_title('Churn Rate: Contract × Tenure', fontsize=12, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.legend(title='Tenure')
ax.tick_params(axis='x', rotation=45)

# MonthlyCharges + InternetService
ax = axes[1, 0]
for service in train['InternetService'].dropna().unique(): # Added .dropna() here
    data = train[train['InternetService'] == service]
    churn_by_charge = data.groupby(pd.cut(data['MonthlyCharges'], bins=10))['Churn'].apply(lambda x: (x == 'Yes').sum() / len(x) * 100)
    ax.plot(range(len(churn_by_charge)), churn_by_charge.values, marker='o', label=service, linewidth=2)
ax.set_title('Churn Rate by Monthly Charges & Internet Service', fontsize=12, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Monthly Charges Bins')
ax.legend()
ax.grid(True, alpha=0.3)

# SeniorCitizen + Contract
ax = axes[1, 1]
senior_labels = {0: 'Not Senior', 1: 'Senior'}
train['SeniorLabel'] = train['SeniorCitizen'].map(senior_labels)
interaction = pd.crosstab([train['SeniorLabel'], train['Contract']],
                          train['Churn'], normalize='index') * 100
interaction['Yes'].unstack().plot(kind='bar', ax=ax, color=['#2ecc71', '#3498db', '#e74c3c'])
ax.set_title('Churn Rate: Senior Status × Contract', fontsize=12, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.legend(title='Contract')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('04_feature_interactions.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 04_feature_interactions.png")
plt.close()

# ============================================================================
# 9. SUMMARY STATISTICS TABLE
# ============================================================================
print("\n" + "=" * 80)
print("5. FEATURE SUMMARY FOR MODELING")
print("=" * 80)

summary = pd.DataFrame({
    'Feature': categorical_cols + numeric_for_viz,
    'Type': ['Categorical'] * len(categorical_cols) + ['Numeric'] * len(numeric_for_viz),
    'Unique_Values': [train[col].nunique() for col in (categorical_cols + numeric_for_viz)],
    'Missing_%': [(train[col].isnull().sum() / len(train) * 100) for col in (categorical_cols + numeric_for_viz)],
})

print(summary.to_string(index=False))

# ============================================================================
# 10. DATA QUALITY CHECKS
# ============================================================================
print("\n" + "=" * 80)
print("6. DATA QUALITY CHECKS")
print("=" * 80)

# Check for TotalCharges issues
print("\nTotalCharges vs MonthlyCharges * tenure:")
train_clean = train.dropna(subset=['TotalCharges', 'MonthlyCharges'])
train_clean['calculated_total'] = train_clean['MonthlyCharges'] * train_clean['tenure']
correlation = train_clean['TotalCharges'].corr(train_clean['calculated_total'])
print(f"Correlation: {correlation:.4f}")

# Rows with nulls
print(f"\nRows with null values:")
null_mask = train['TotalCharges'].isnull() | train['MonthlyCharges'].isnull()
if null_mask.sum() > 0:
    print(train[null_mask][['id', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])
else:
    print("No null values found!")

print("\n" + "=" * 80)
print("✓ ALL VISUALIZATIONS COMPLETE!")
print("=" * 80)
print("\nGenerated files:")
print("  1. 01_churn_overview.png - Overall churn distribution & top features")
print("  2. 02_numeric_vs_churn.png - Numeric features by churn status")
print("  3. 03_all_categorical_churn_rates.png - All categorical features")
print("  4. 04_feature_interactions.png - Key feature interactions")
print("\nNext: Share these insights, and we'll build the ensemble model!")

LOADING DATA...
✓ Loaded 594194 rows × 21 columns

1. DATASET OVERVIEW
Shape: (594194, 21)

Data types:
id                    int64
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

Missing values:
id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport  

In [ ]:
"""
ADVANCED FEATURE ENGINEERING FOR CUSTOMER CHURN
==============================================

This script demonstrates all feature engineering techniques to maximize model performance.
Key strategies:
1. Domain-driven features (based on EDA insights)
2. Statistical features (interactions, ratios)
3. Temporal patterns (tenure-based features)
4. Categorical encoding (smart encoding based on churn rates)
5. Polynomial & interaction features

Copy and paste this entire code into your environment.
Make sure train.csv is in the same directory.
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("FEATURE ENGINEERING FOR CUSTOMER CHURN")
print("=" * 80)

# ============================================================================
# 1. LOAD & PREPARE DATA
# ============================================================================
print("\n[STEP 1] Loading data...")
train = pd.read_csv('train.csv')
print(f"✓ Loaded {len(train)} rows × {len(train.columns)} columns")

# Handle the single null row
train = train.dropna(subset=['TotalCharges', 'MonthlyCharges', 'Churn'])
print(f"✓ After cleaning nulls: {len(train)} rows")

# Separate features and target
X = train.drop(['id', 'Churn'], axis=1)
y = (train['Churn'] == 'Yes').astype(int)

print(f"✓ Target distribution: {y.value_counts().to_dict()}")
print(f"✓ Churn rate: {y.mean()*100:.2f}%\n")

# ============================================================================
# 2. IDENTIFY FEATURE TYPES
# ============================================================================
print("[STEP 2] Identifying feature types...")

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"✓ Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"✓ Categorical features ({len(categorical_features)}): {categorical_features}\n")

# ============================================================================
# 3. ENGINEER NUMERIC FEATURES
# ============================================================================
print("[STEP 3] Engineering numeric features...")
X_numeric = X[numeric_features].copy()

# 3a. Tenure-based features (KEY INSIGHT: tenure is strongest churn predictor)
X_numeric['tenure_squared'] = X_numeric['tenure'] ** 2
X_numeric['tenure_log'] = np.log1p(X_numeric['tenure'])  # Log for non-linear relationships
X_numeric['tenure_is_new'] = (X_numeric['tenure'] <= 12).astype(int)  # New customer flag
X_numeric['tenure_is_medium'] = ((X_numeric['tenure'] > 12) & (X_numeric['tenure'] <= 36)).astype(int)
X_numeric['tenure_is_loyal'] = (X_numeric['tenure'] > 36).astype(int)

# 3b. Charge-based features
X_numeric['charge_per_month'] = X_numeric['MonthlyCharges'] / X_numeric['tenure'].replace(0, 1)
X_numeric['MonthlyCharges_log'] = np.log1p(X_numeric['MonthlyCharges'])
X_numeric['TotalCharges_log'] = np.log1p(X_numeric['TotalCharges'])
X_numeric['high_charges_flag'] = (X_numeric['MonthlyCharges'] > X_numeric['MonthlyCharges'].median()).astype(int)

# 3c. Charge ratio features
X_numeric['charge_ratio'] = X_numeric['TotalCharges'] / X_numeric['MonthlyCharges'].replace(0, 1)

print("✓ Created tenure features: tenure_squared, tenure_log, tenure_is_new, tenure_is_medium, tenure_is_loyal")
print("✓ Created charge features: charge_per_month, MonthlyCharges_log, TotalCharges_log, high_charges_flag")
print("✓ Created ratio features: charge_ratio\n")

# ============================================================================
# 4. TARGET ENCODING FOR CATEGORICAL FEATURES
# ============================================================================
print("[STEP 4] Encoding categorical features...")

X_categorical = X[categorical_features].copy()

# STRATEGY 1: Target Encoding for high-cardinality features
# Based on churn probability for each category
def target_encode(feature, target):
    """
    Map each category to its churn rate.
    High-churn categories get high values, low-churn get low values.
    This gives the model a strong signal.
    """
    encoding_map = target.groupby(feature).mean()
    return feature.map(encoding_map)

# Apply target encoding to key features (strong churn signals)
target_encode_features = [
    'Contract',           # 42.7% vs 1%: HUGE difference
    'PaymentMethod',      # 49.6% vs 7%: HUGE difference
    'InternetService',    # 42.3% vs 1.6%: HUGE difference
]

X_categorical_encoded = X_categorical.copy()

for feature in target_encode_features:
    X_categorical_encoded[f'{feature}_target_encoded'] = target_encode(X_categorical[feature], y)
    print(f"✓ Target encoded: {feature}")

print()

# STRATEGY 2: One-Hot Encoding for other features (binary/3-level features)
# These don't have huge churn variance, so one-hot is fine
onehot_features = [col for col in categorical_features if col not in target_encode_features]

# Drop the original features after encoding
X_categorical_drop = X_categorical.drop(target_encode_features, axis=1)

# One-hot encode
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
X_onehot = encoder.fit_transform(X_categorical_drop)
X_onehot_df = pd.DataFrame(X_onehot, columns=encoder.get_feature_names_out(onehot_features))

print(f"✓ One-hot encoded {len(onehot_features)} features: {onehot_features}")
print(f"✓ Created {X_onehot_df.shape[1]} one-hot encoded columns\n")

# ============================================================================
# 5. FEATURE INTERACTIONS (Domain-Driven)
# ============================================================================
print("[STEP 5] Creating feature interactions...")

# From EDA: Contract × Tenure interaction is HUGE
# Month-to-month + new customer = EXTREME churn risk
X_numeric['contract_tenure_risk'] = (
    X_categorical_encoded['Contract_target_encoded'] *
    X_numeric['tenure_is_new']
)

# Fiber optic + high charges = high churn
X_numeric['fiber_high_charge'] = (
    X_categorical_encoded['InternetService_target_encoded'] *
    X_numeric['high_charges_flag']
)

# No tech support + high charges = high churn
X_numeric['no_support_high_charge'] = (
    (X_categorical[['TechSupport']] == 'No').astype(int).values.flatten() *
    X_numeric['high_charges_flag']
)

print("✓ Created contract_tenure_risk: Contract × New Customer")
print("✓ Created fiber_high_charge: Fiber Optic × High Charges")
print("✓ Created no_support_high_charge: No Support × High Charges\n")

# ============================================================================
# 6. ADD-ON FEATURES (Based on internet services)
# ============================================================================
print("[STEP 6] Creating add-on features...")

addon_features = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count how many add-ons customer has (excluding "No internet service")
X_numeric['num_addons'] = 0
for addon in addon_features:
    X_numeric['num_addons'] += (X_categorical[addon] == 'Yes').astype(int)

X_numeric['has_security_backup'] = (
    ((X_categorical['OnlineSecurity'] == 'Yes').astype(int) &
     (X_categorical['OnlineBackup'] == 'Yes').astype(int)).astype(int)
)

X_numeric['has_tech_support'] = (X_categorical['TechSupport'] == 'Yes').astype(int)

print(f"✓ Created num_addons: count of security/support add-ons")
print(f"✓ Created has_security_backup: both security AND backup flag")
print(f"✓ Created has_tech_support: tech support flag\n")

# ============================================================================
# 7. SENIOR & FAMILY STATUS FEATURES
# ============================================================================
print("[STEP 7] Creating demographic features...")

X_numeric['is_senior'] = X['SeniorCitizen'].copy()
X_numeric['has_partner'] = (X_categorical['Partner'] == 'Yes').astype(int)
X_numeric['has_dependents'] = (X_categorical['Dependents'] == 'Yes').astype(int)

# Family status: high protection factor
X_numeric['family_size_proxy'] = X_numeric['has_partner'] + X_numeric['has_dependents']

# Senior with no support is high risk
X_numeric['senior_no_support'] = (
    X_numeric['is_senior'] *
    (1 - X_numeric['has_tech_support'])
)

print("✓ Created is_senior, has_partner, has_dependents")
print("✓ Created family_size_proxy: combined partner + dependents")
print("✓ Created senior_no_support: senior without tech support risk\n")

# ============================================================================
# 8. COMBINE ALL FEATURES
# ============================================================================
print("[STEP 8] Combining all engineered features...")

# Combine numeric and categorical
X_engineered = pd.concat([
    X_numeric,  # All engineered numeric features
    X_categorical_encoded[target_encode_features + [f'{f}_target_encoded' for f in target_encode_features]],  # Target-encoded features
    X_onehot_df  # One-hot encoded features
], axis=1)

# Remove original non-encoded categorical columns
X_engineered = X_engineered.drop([col for col in X_categorical_encoded.columns if col not in
                                   target_encode_features + [f'{f}_target_encoded' for f in target_encode_features]], axis=1, errors='ignore')

print(f"✓ Total engineered features: {X_engineered.shape[1]}")
print(f"✓ Final dataset shape: {X_engineered.shape}\n")

# ============================================================================
# 9. FEATURE STATISTICS & INSIGHTS
# ============================================================================
print("[STEP 9] Feature Engineering Summary")
print("=" * 80)

print("\nNUMERIC FEATURE ENGINEERING:")
print("-" * 80)
numeric_engineered_features = [col for col in X_numeric.columns if col not in numeric_features]
print(f"Original numeric features: {numeric_features}")
print(f"\nEngineered numeric features ({len(numeric_engineered_features)}):")
for i, feat in enumerate(numeric_engineered_features, 1):
    print(f"  {i}. {feat}")

print("\n\nCATEGORICAL FEATURE ENGINEERING:")
print("-" * 80)
print(f"Target-encoded features: {target_encode_features}")
for feat in target_encode_features:
    churn_rates = y.groupby(X[feat]).mean().sort_values(ascending=False)
    print(f"\n  {feat}:")
    for cat, rate in churn_rates.items():
        print(f"    {cat}: {rate*100:.1f}% churn")

print(f"\n\nOne-hot encoded features ({X_onehot_df.shape[1]} total):")
print(f"  From: {onehot_features}")

# ============================================================================
# 10. SAVE ENGINEERED DATA
# ============================================================================
print("\n" + "=" * 80)
print("[STEP 10] Saving engineered features...")

# Create output dataframe with engineered features + target
output_df = X_engineered.copy()
output_df['Churn'] = y

# Save to CSV for next step
output_df.to_csv('train_engineered.csv', index=False)
print(f"✓ Saved: train_engineered.csv ({output_df.shape[0]} rows × {output_df.shape[1]} columns)")

# Also save feature names and types for easy reference
feature_info = pd.DataFrame({
    'feature': X_engineered.columns,
    'type': ['numeric'] * X_numeric.shape[1] + ['categorical'] * (X_engineered.shape[1] - X_numeric.shape[1])
})
feature_info.to_csv('feature_info.csv', index=False)
print(f"✓ Saved: feature_info.csv")

# ============================================================================
# 11. SUMMARY STATISTICS
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 80)
print(f"\nOriginal features: {len(X.columns)}")
print(f"Engineered features: {X_engineered.shape[1]}")
print(f"Feature expansion: {X_engineered.shape[1] / len(X.columns):.2f}x")
print(f"\nFeature breakdown:")
print(f"  - Numeric (original + derived): {X_numeric.shape[1]}")
print(f"  - Categorical (target-encoded): {len(target_encode_features) * 2}")
print(f"  - Categorical (one-hot): {X_onehot_df.shape[1]}")
print(f"\nKey engineered features (highest impact):")
print(f"  1. Contract (target encoded) - 42.7% vs 1% churn difference")
print(f"  2. PaymentMethod (target encoded) - 49.6% vs 7% churn difference")
print(f"  3. InternetService (target encoded) - 42.3% vs 1.6% churn difference")
print(f"  4. tenure_is_new - captures critical 0-1 year risk period")
print(f"  5. contract_tenure_risk - interaction of contract × new customer")
print(f"  6. num_addons - count of protective factors")
print(f"  7. family_size_proxy - customers with partner/dependents churn less")

print("\n" + "=" * 80)
print("✓ FEATURE ENGINEERING COMPLETE!")
print("=" * 80)
print("\nNext steps:")
print("  1. Review the engineered features above")
print("  2. Run this script to create train_engineered.csv")
print("  3. Use train_engineered.csv for model building")
print("\nDataset ready for ensemble modeling!")

FEATURE ENGINEERING FOR CUSTOMER CHURN

[STEP 1] Loading data...
✓ Loaded 594194 rows × 21 columns
✓ After cleaning nulls: 594194 rows
✓ Target distribution: {0: 460377, 1: 133817}
✓ Churn rate: 22.52%

[STEP 2] Identifying feature types...
✓ Numeric features (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
✓ Categorical features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

[STEP 3] Engineering numeric features...
✓ Created tenure features: tenure_squared, tenure_log, tenure_is_new, tenure_is_medium, tenure_is_loyal
✓ Created charge features: charge_per_month, MonthlyCharges_log, TotalCharges_log, high_charges_flag
✓ Created ratio features: charge_ratio

[STEP 4] Encoding categorical features...
✓ Target encoded: Contract
✓ Target encoded: PaymentMethod
✓ Target encoded

In [ ]:
"""
FIXED FEATURE REDUNDANCY ANALYSIS
=================================

This script ACTUALLY removes redundant features correctly:
- tenure vs tenure_log: keeps ONLY tenure_log (better performance)
- MonthlyCharges vs MonthlyCharges_log: keeps ONLY MonthlyCharges_log
- Removes duplicate one-hot encoded "No" variants
- Removes all "No internet service" variants
- Aggressively removes near-duplicates

For Google Colab
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("FIXED FEATURE REDUNDANCY ANALYSIS")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD DATA
# ============================================================================
print("\n[STEP 1] Loading data...")
df = pd.read_csv('train_engineered.csv')

X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"✓ Loaded {X.shape[0]} rows × {X.shape[1]} features")
print(f"✓ Target: {y.value_counts().to_dict()}\n")

# ============================================================================
# STEP 2: IDENTIFY FEATURES TO DROP - EXPLICIT LIST
# ============================================================================
print("[STEP 2] Identifying features to drop...\n")

features_to_drop = set()

# Get numeric columns and their churn correlations
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
y_encoded = (y == 1).astype(int)

feature_corr = {}
for col in numeric_cols:
    feature_corr[col] = abs(X[col].corr(pd.Series(y_encoded)))

# RULE 1: REMOVE RAW FEATURES WHEN LOG VARIANT EXISTS
# ===================================================
print("RULE 1: Removing raw features (keeping log variants only)...\n")

log_replacements = {
    'tenure': 'tenure_log',
    'MonthlyCharges': 'MonthlyCharges_log',
    'TotalCharges': 'TotalCharges_log'
}

for raw_feat, log_feat in log_replacements.items():
    if raw_feat in X.columns and log_feat in X.columns:
        features_to_drop.add(raw_feat)
        raw_corr = feature_corr.get(raw_feat, 0)
        log_corr = feature_corr.get(log_feat, 0)
        print(f"  DROP: {raw_feat:35s} (corr: {raw_corr:.4f})")
        print(f"  KEEP: {log_feat:35s} (corr: {log_corr:.4f}) ← BETTER\n")

# RULE 2: REMOVE REDUNDANT TENURE VARIANTS
# =========================================
print("RULE 2: Removing redundant tenure engineered variants...\n")

tenure_variants = {
    'tenure_squared': feature_corr.get('tenure_squared', 0),
    'tenure_is_new': feature_corr.get('tenure_is_new', 0),
    'tenure_is_medium': feature_corr.get('tenure_is_medium', 0),
    'tenure_is_loyal': feature_corr.get('tenure_is_loyal', 0),
}

# Only keep tenure_log (already kept, others drop)
for var, corr in tenure_variants.items():
    if var in X.columns:
        features_to_drop.add(var)
        print(f"  DROP: {var:35s} (corr: {corr:.4f})")

print(f"  KEEP: tenure_log (ONLY tenure variant)\n")

# RULE 3: REMOVE REDUNDANT CHARGES VARIANTS
# ==========================================
print("RULE 3: Removing redundant charge engineered variants...\n")

# Keep only best 2: MonthlyCharges_log and charge_per_month
charge_variants_drop = {
    'high_charges_flag': feature_corr.get('high_charges_flag', 0),
}

for var, corr in charge_variants_drop.items():
    if var in X.columns:
        features_to_drop.add(var)
        print(f"  DROP: {var:35s} (corr: {corr:.4f})")

print(f"  KEEP: MonthlyCharges_log (corr: {feature_corr.get('MonthlyCharges_log', 0):.4f})")
print(f"  KEEP: charge_per_month (corr: {feature_corr.get('charge_per_month', 0):.4f})\n")

# RULE 4: REMOVE ONE-HOT ENCODED COMPLEMENTS
# ===========================================
print("RULE 4: Removing one-hot encoded '_No' variants (keeping '_Yes' only)...\n")

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# Find all _Yes and _No pairs
yes_no_pairs = {}
for col in numeric_cols:
    if col.endswith('_Yes'):
        base = col[:-4]  # Remove '_Yes'
        no_variant = f"{base}_No"
        if no_variant in X.columns:
            yes_no_pairs[col] = no_variant

# Drop all _No variants (they're complements of _Yes)
for yes_col, no_col in yes_no_pairs.items():
    if no_col not in features_to_drop:
        features_to_drop.add(no_col)
        yes_corr = feature_corr.get(yes_col, 0)
        no_corr = feature_corr.get(no_col, 0)
        print(f"  DROP: {no_col:35s} (corr: {no_corr:.4f})")
        print(f"  KEEP: {yes_col:35s} (corr: {yes_corr:.4f}) ← complement\n")

# RULE 5: REMOVE ALL "NO INTERNET SERVICE" VARIANTS
# =================================================
print("RULE 5: Removing 'No internet service' one-hot variants...\n")

no_internet_variants = [col for col in numeric_cols if 'No internet service' in col]

for col in no_internet_variants:
    if col not in features_to_drop:
        features_to_drop.add(col)
        corr = feature_corr.get(col, 0)
        print(f"  DROP: {col:35s} (corr: {corr:.4f})")

print(f"  Reason: If no internet, all add-ons = 'No internet service' (inverse signal)\n")

# RULE 6: REMOVE EXACT DUPLICATES (correlation = 1.0)
# ==================================================
print("RULE 6: Removing exact duplicates (corr = 1.0)...\n")

X_numeric = X[numeric_cols].copy()
corr_matrix = X_numeric.corr()

# Find pairs with correlation = 1.0
duplicates_found = False
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        if abs(corr_matrix.iloc[i, j] - 1.0) < 0.001:  # Allow small numerical error
            feat1 = numeric_cols[i]
            feat2 = numeric_cols[j]

            # Skip if already marked for drop
            if feat1 in features_to_drop or feat2 in features_to_drop:
                continue

            corr1 = feature_corr.get(feat1, 0)
            corr2 = feature_corr.get(feat2, 0)

            # Drop the one with lower churn correlation
            if corr1 > corr2:
                features_to_drop.add(feat2)
                print(f"  DROP: {feat2:35s} (corr with {feat1}: 1.0000)")
                print(f"  KEEP: {feat1:35s} (churn_corr: {corr1:.4f})\n")
            else:
                features_to_drop.add(feat1)
                print(f"  DROP: {feat1:35s} (corr with {feat2}: 1.0000)")
                print(f"  KEEP: {feat2:35s} (churn_corr: {corr2:.4f})\n")

            duplicates_found = True

if not duplicates_found:
    print("  ✓ No exact duplicates found\n")

# RULE 7: DROP contract_tenure_risk IF keeping tenure_log
# =====================================================
print("RULE 7: Smart interaction feature selection...\n")

# contract_tenure_risk = Contract_encoded × tenure_is_new
# We're already keeping tenure_log, so this interaction is redundant
if 'contract_tenure_risk' in X.columns and 'tenure_is_new' in features_to_drop:
    if 'contract_tenure_risk' not in features_to_drop:
        features_to_drop.add('contract_tenure_risk')
        print(f"  DROP: contract_tenure_risk (redundant with tenure_log)")
        print(f"  REASON: tenure_is_new already dropped\n")

# ============================================================================
# STEP 3: CREATE FINAL CLEAN DATASET
# ============================================================================
print("[STEP 3] Creating final clean dataset...\n")

features_to_keep = [col for col in X.columns if col not in features_to_drop]

X_clean = X[features_to_keep].copy()
df_clean = X_clean.copy()
df_clean['Churn'] = y.values

print(f"REDUCTION SUMMARY:")
print(f"  Original features: {X.shape[1]}")
print(f"  Features to drop: {len(features_to_drop)}")
print(f"  Final features: {X_clean.shape[1]}")
print(f"  Reduction: {len(features_to_drop)/X.shape[1]*100:.1f}%\n")

# Save
df_clean.to_csv('train_final_clean.csv', index=False)
print(f"✓ Saved: train_final_clean.csv ({df_clean.shape})\n")

# ============================================================================
# STEP 4: FEATURE STATISTICS
# ============================================================================
print("[STEP 4] Final feature statistics...\n")

numeric_final = X_clean.select_dtypes(include=[np.number]).columns.tolist()

# Calculate churn correlations for final features
final_feature_corr = {}
for col in numeric_final:
    final_feature_corr[col] = abs(X_clean[col].corr(pd.Series(y_encoded)))

final_df = pd.DataFrame(list(final_feature_corr.items()), columns=['feature', 'churn_correlation'])
final_df = final_df.sort_values('churn_correlation', ascending=False)

print("TOP 15 FINAL FEATURES:")
for i, row in final_df.head(15).iterrows():
    print(f"  {row['feature']:40s} : {row['churn_correlation']:.4f}")

print("\n")

# ============================================================================
# STEP 5: VERIFY NO REDUNDANCY
# ============================================================================
print("[STEP 5] Verifying no remaining redundancy...\n")

X_clean_numeric = X_clean[numeric_final].copy()
corr_clean = X_clean_numeric.corr()

# Check for high correlations
high_corr_count = 0
for i in range(len(numeric_final)):
    for j in range(i + 1, len(numeric_final)):
        corr_val = abs(corr_clean.iloc[i, j])
        if corr_val > 0.90:
            high_corr_count += 1
            print(f"  {numeric_final[i]:35s} ↔ {numeric_final[j]:35s} : {corr_val:.4f}")

if high_corr_count == 0:
    print("✓ No feature pairs with |correlation| > 0.90")
else:
    print(f"⚠ Found {high_corr_count} pairs with high correlation (may need further cleaning)")

print("\n")

# ============================================================================
# STEP 6: CREATE VISUALIZATIONS
# ============================================================================
print("[STEP 6] Creating visualizations...\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Correlation heatmap
ax = axes[0, 0]
sns.heatmap(corr_clean, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax, annot=False)
ax.set_title('Final Feature Correlation Matrix\n(Clean - No Redundancy)', fontsize=12, fontweight='bold')

# 2. Feature importance
ax = axes[0, 1]
top_20 = final_df.head(20)
ax.barh(range(len(top_20)), top_20['churn_correlation'], color='#2ecc71', edgecolor='black')
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['feature'], fontsize=9)
ax.set_xlabel('Churn Correlation', fontsize=10, fontweight='bold')
ax.set_title('Top 20 Final Features', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# 3. Feature reduction breakdown
ax = axes[1, 0]
stages = ['Original\nEngineered', 'Removed\nRedundancy', 'Final\nClean']
counts = [X.shape[1], len(features_to_drop), X_clean.shape[1]]
colors = ['#3498db', '#e74c3c', '#2ecc71']
bars = ax.bar(stages, counts, color=colors, edgecolor='black', linewidth=2, width=0.6)
ax.set_ylabel('Number of Features', fontsize=10, fontweight='bold')
ax.set_title('Feature Count Reduction', fontsize=12, fontweight='bold')
for bar, val in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{int(val)}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 4. Removal reasons pie chart
ax = axes[1, 1]
removal_counts = {
    'Log variants': 3,
    'Tenure variants': len([f for f in features_to_drop if 'tenure' in f.lower() and f != 'tenure']),
    'One-hot complements': len(yes_no_pairs),
    'No internet service': len(no_internet_variants),
    'Other': len(features_to_drop) - 3 - len([f for f in features_to_drop if 'tenure' in f.lower()]) - len(yes_no_pairs) - len(no_internet_variants)
}
removal_counts = {k: v for k, v in removal_counts.items() if v > 0}
colors_pie = ['#e74c3c', '#f39c12', '#e67e22', '#c0392b', '#95a5a6']
ax.pie(removal_counts.values(), labels=removal_counts.keys(), autopct='%1.1f%%',
       colors=colors_pie, startangle=90)
ax.set_title(f'Removed Features Distribution\n(Total: {len(features_to_drop)})', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('final_clean_summary.png', dpi=150, bbox_inches='tight')
print("✓ Saved: final_clean_summary.png\n")
plt.close()

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"\nFeature Cleanup Results:")
print(f"  Starting features: {X.shape[1]}")
print(f"  Removed: {len(features_to_drop)}")
print(f"  Final: {X_clean.shape[1]}")
print(f"  Reduction: {len(features_to_drop)/X.shape[1]*100:.1f}%")

print(f"\nKey Removals:")
print(f"  ✗ tenure (raw)")
print(f"  ✗ MonthlyCharges (raw)")
print(f"  ✗ TotalCharges (raw)")
print(f"  ✗ All tenure variant (squared, is_new, is_medium, is_loyal)")
print(f"  ✗ One-hot '_No' variants (kept '_Yes' only)")
print(f"  ✗ All 'No internet service' variants")
print(f"  ✗ contract_tenure_risk (if tenure_is_new dropped)")

print(f"\nKey Kept Features:")
print(f"  ✓ tenure_log (ONLY tenure variant)")
print(f"  ✓ MonthlyCharges_log")
print(f"  ✓ charge_per_month")
print(f"  ✓ TotalCharges_log")
print(f"  ✓ Contract_target_encoded")
print(f"  ✓ PaymentMethod_target_encoded")
print(f"  ✓ InternetService_target_encoded")
print(f"  ✓ no_support_high_charge")
print(f"  ✓ fiber_high_charge")
print(f"  ✓ num_addons, has_security_backup, family_size_proxy")
print(f"  ✓ Plus demographic and service features (one-hot _Yes variants)")

print(f"\nNext Step: Use train_final_clean.csv for ensemble modeling!")
print("=" * 80)


FIXED FEATURE REDUNDANCY ANALYSIS

[STEP 1] Loading data...
✓ Loaded 594194 rows × 50 features
✓ Target: {0: 460377, 1: 133817}

[STEP 2] Identifying features to drop...

RULE 1: Removing raw features (keeping log variants only)...

  DROP: tenure                              (corr: 0.4185)
  KEEP: tenure_log                          (corr: 0.4379) ← BETTER

  DROP: MonthlyCharges                      (corr: 0.2730)
  KEEP: MonthlyCharges_log                  (corr: 0.2894) ← BETTER

  DROP: TotalCharges                        (corr: 0.2184)
  KEEP: TotalCharges_log                    (corr: 0.2482) ← BETTER

RULE 2: Removing redundant tenure engineered variants...

  DROP: tenure_squared                      (corr: 0.3784)
  DROP: tenure_is_new                       (corr: 0.3783)
  DROP: tenure_is_medium                    (corr: 0.0369)
  DROP: tenure_is_loyal                     (corr: 0.3629)
  KEEP: tenure_log (ONLY tenure variant)

RULE 3: Removing redundant charge engineered va

In [ ]:
!pip install xgboost lightgbm catboost tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.9 MB/s eta 0:00:00


In [ ]:

"""
PREPROCESSING PIPELINE
======================

Single function that handles ALL data preprocessing:
- Feature engineering
- Redundancy removal
- Encoding
- Scaling

Can be applied to train OR test data consistently.

For Google Colab - copy this cell
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
import pickle
import warnings
warnings.filterwarnings('ignore')

def preprocess_pipeline(df, fit_encoders=False, encoders_dict=None):
    """
    Complete preprocessing pipeline for churn data.

    Parameters:
    -----------
    df : pd.DataFrame
        Raw input dataframe (train or test)
    fit_encoders : bool
        If True, fit and return new encoders (use on train data)
        If False, use existing encoders (use on test data)
    encoders_dict : dict
        Dictionary of fitted encoders from training data
        Required if fit_encoders=False

    Returns:
    --------
    X : pd.DataFrame
        Processed features
    y : pd.Series or None
        Target variable (None if not in input)
    encoders : dict
        Dictionary of fitted encoders (for use on test data)
    """

    print("=" * 80)
    print("PREPROCESSING PIPELINE")
    print("=" * 80)

    df = df.copy()

    # ========================================================================
    # STEP 0: EXTRACT TARGET (if exists)
    # ========================================================================
    y = None
    if 'Churn' in df.columns:
        y = (df['Churn'] == 'Yes').astype(int)
        df = df.drop(['Churn', 'id'], axis=1, errors='ignore')
        print(f"\n✓ Extracted target variable")
        print(f"  Shape: {df.shape}")
    else:
        df = df.drop(['id'], axis=1, errors='ignore')
        print(f"\n✓ No target variable (test data)")
        print(f"  Shape: {df.shape}")

    # ========================================================================
    # STEP 1: IDENTIFY FEATURE TYPES
    # ========================================================================
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = df.select_dtypes(include=['object']).columns.tolist()

    print(f"\n✓ Numeric features: {len(numeric_features)}")
    print(f"✓ Categorical features: {len(categorical_features)}")

    # ========================================================================
    # STEP 2: ENGINEER NUMERIC FEATURES
    # ========================================================================
    print(f"\n[STEP 1] Engineering numeric features...")

    X_numeric = df[numeric_features].copy()

    # Handle nulls
    X_numeric = X_numeric.fillna(X_numeric.median())

    # Tenure-based features
    X_numeric['tenure_log'] = np.log1p(X_numeric['tenure'])

    # Charge-based features
    X_numeric['charge_per_month'] = X_numeric['MonthlyCharges'] / X_numeric['tenure'].replace(0, 1)
    X_numeric['MonthlyCharges_log'] = np.log1p(X_numeric['MonthlyCharges'])
    X_numeric['TotalCharges_log'] = np.log1p(X_numeric['TotalCharges'])
    X_numeric['charge_ratio'] = X_numeric['TotalCharges'] / X_numeric['MonthlyCharges'].replace(0, 1)

    # Keep only best numeric features (no redundancy)
    numeric_features_keep = [
        'tenure_log',  # ONLY tenure variant
        'MonthlyCharges_log',
        'charge_per_month',
        'TotalCharges_log',
        'charge_ratio'
    ]

    X_numeric = X_numeric[numeric_features_keep]
    print(f"✓ Engineered numeric features: {list(X_numeric.columns)}")

    # ========================================================================
    # STEP 3: ENCODE CATEGORICAL FEATURES
    # ========================================================================
    print(f"\n[STEP 2] Encoding categorical features...")

    X_categorical = df[categorical_features].copy()

    # Target encoding for high-impact features
    target_encode_features = ['Contract', 'PaymentMethod', 'InternetService']

    if fit_encoders:
        # TRAINING MODE: Fit encoders
        print(f"  Mode: FITTING new encoders")
        encoders_dict = {
            'target_encoders': {},
            'onehot_encoder': None,
            'scaler': None
        }

        for feature in target_encode_features:
            if feature in X_categorical.columns:
                if y is not None:
                    encoding_map = y.groupby(df[feature]).mean().to_dict()
                    X_categorical[f'{feature}_encoded'] = X_categorical[feature].map(encoding_map)
                    encoders_dict['target_encoders'][feature] = encoding_map
                    print(f"    ✓ Target encoded: {feature}")

        # One-hot encode remaining features (keep _Yes only, drop _No)
        onehot_features = [col for col in categorical_features if col not in target_encode_features]

        # Fit encoder
        onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
        X_onehot = onehot_encoder.fit_transform(X_categorical[onehot_features])
        X_onehot_df = pd.DataFrame(X_onehot, columns=onehot_encoder.get_feature_names_out(onehot_features))

        encoders_dict['onehot_encoder'] = onehot_encoder
        print(f"    ✓ One-hot encoded: {onehot_features}")

    else:
        # TEST MODE: Use existing encoders
        print(f"  Mode: USING existing encoders")

        for feature in target_encode_features:
            if feature in X_categorical.columns:
                encoding_map = encoders_dict['target_encoders'].get(feature, {})
                X_categorical[f'{feature}_encoded'] = X_categorical[feature].map(encoding_map)
                # Fill unknown values with mean (0.5 for binary churn encoding)
                X_categorical[f'{feature}_encoded'] = X_categorical[f'{feature}_encoded'].fillna(0.5)
                print(f"    ✓ Target encoded: {feature}")

        # One-hot encode
        onehot_features = [col for col in categorical_features if col not in target_encode_features]
        X_onehot = encoders_dict['onehot_encoder'].transform(X_categorical[onehot_features])
        X_onehot_df = pd.DataFrame(X_onehot, columns=encoders_dict['onehot_encoder'].get_feature_names_out(onehot_features))
        print(f"    ✓ One-hot encoded: {onehot_features}")

    # Extract only encoded target features
    X_categorical_encoded = X_categorical[[f'{f}_encoded' for f in target_encode_features if f in X_categorical.columns]]

    # ========================================================================
    # STEP 4: CREATE INTERACTION & DERIVED FEATURES
    # ========================================================================
    print(f"\n[STEP 3] Creating interaction features...")

    # Interaction: No tech support + high charges
    has_tech_support = (df['TechSupport'] == 'Yes').astype(int)
    high_charges = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)
    X_numeric['no_support_high_charge'] = (1 - has_tech_support) * high_charges

    # Interaction: Fiber optic + high charges
    is_fiber = (df['InternetService'] == 'Fiber optic').astype(int)
    X_numeric['fiber_high_charge'] = is_fiber * high_charges

    # Count of add-ons
    addon_features = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                      'TechSupport', 'StreamingTV', 'StreamingMovies']
    X_numeric['num_addons'] = sum([(df[addon] == 'Yes').astype(int) for addon in addon_features if addon in df.columns])

    # Family proxy
    X_numeric['family_size_proxy'] = (
        (df['Partner'] == 'Yes').astype(int) +
        (df['Dependents'] == 'Yes').astype(int)
    )

    # Security backup
    X_numeric['has_security_backup'] = (
        ((df['OnlineSecurity'] == 'Yes').astype(int)) *
        ((df['OnlineBackup'] == 'Yes').astype(int))
    )

    # Senior status
    X_numeric['senior_no_support'] = (
        (df['SeniorCitizen'] == 1).astype(int) *
        (1 - has_tech_support)
    )

    print(f"✓ Created interaction features: no_support_high_charge, fiber_high_charge")
    print(f"✓ Created count features: num_addons, family_size_proxy, has_security_backup, senior_no_support")

    # ========================================================================
    # STEP 5: ONE-HOT ENCODE SELECT CATEGORICAL
    # ========================================================================
    print(f"\n[STEP 4] One-hot encoding categorical features...")

    # Keep one-hot for: gender, Partner, Dependents, PhoneService, etc.
    # But remove _No variants (keep only _Yes)
    categorical_for_onehot = [
        'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
    ]

    # Remove duplicates/complements, keep only meaningful variants
    X_categorical_final = pd.DataFrame()

    for feature in categorical_for_onehot:
        if feature in df.columns:
            # For binary features, create _Yes variant only
            X_categorical_final[f'{feature}_Yes'] = (df[feature] == 'Yes').astype(int)

    print(f"✓ Created one-hot features: {list(X_categorical_final.columns)}")

    # ========================================================================
    # STEP 6: COMBINE ALL FEATURES
    # ========================================================================
    print(f"\n[STEP 5] Combining all features...")

    X = pd.concat([
        X_numeric,
        X_categorical_encoded,
        X_categorical_final
    ], axis=1)

    print(f"✓ Total features: {X.shape[1]}")
    print(f"✓ Final shape: {X.shape}")

    # ========================================================================
    # STEP 7: SCALE NUMERIC FEATURES
    # ========================================================================
    print(f"\n[STEP 6] Scaling numeric features...")

    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    if fit_encoders:
        scaler = StandardScaler()
        X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
        encoders_dict['scaler'] = scaler
        print(f"✓ Fitted scaler")
    else:
        X[numeric_cols] = encoders_dict['scaler'].transform(X[numeric_cols])
        print(f"✓ Applied existing scaler")

    print(f"\n✓ PREPROCESSING COMPLETE")
    print(f"  Final shape: {X.shape}")
    if y is not None:
        print(f"  Target distribution: {y.value_counts().to_dict()}")
    print("=" * 80)

    return X, y, encoders_dict


# ============================================================================
# EXAMPLE USAGE (uncomment to test)
# ============================================================================


In [ ]:
!pip install optuna xgboost lightgbm catboost tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.3 MB/s eta 0:00:00


In [ ]:

# STEP 1: PREPROCESS TRAINING DATA
train_raw = pd.read_csv('train.csv')
X_train, y_train, encoders = preprocess_pipeline(train_raw, fit_encoders=True)

# STEP 2: PREPROCESS TEST DATA (use same encoders) - NO TARGET VARIABLE
test_raw = pd.read_csv('test.csv')
X_test, _, _ = preprocess_pipeline(test_raw, fit_encoders=False, encoders_dict=encoders)

# STEP 3: SPLIT TRAINING DATA INTO TRAIN & VALIDATION
from sklearn.model_selection import train_test_split

X_train_tuned, X_val, y_train_tuned, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"\nTraining set: {X_train_tuned.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set (unlabeled): {X_test.shape}")
print(f"All have same features: {list(X_train_tuned.columns) == list(X_val.columns) == list(X_test.columns)}")


PREPROCESSING PIPELINE

✓ Extracted target variable
  Shape: (594194, 19)

✓ Numeric features: 4
✓ Categorical features: 15

[STEP 1] Engineering numeric features...
✓ Engineered numeric features: ['tenure_log', 'MonthlyCharges_log', 'charge_per_month', 'TotalCharges_log', 'charge_ratio']

[STEP 2] Encoding categorical features...
  Mode: FITTING new encoders
    ✓ Target encoded: Contract
    ✓ Target encoded: PaymentMethod
    ✓ Target encoded: InternetService
    ✓ One-hot encoded: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

[STEP 3] Creating interaction features...
✓ Created interaction features: no_support_high_charge, fiber_high_charge
✓ Created count features: num_addons, family_size_proxy, has_security_backup, senior_no_support

[STEP 4] One-hot encoding categorical features...
✓ Created one-hot features: ['gender_Yes', 'Partner_Ye

In [ ]:
"""
TRAIN/VALIDATION/TEST SPLIT SETUP
=================================

Run this cell after preprocessing to properly split data

This cell:
1. Takes X_train, y_train (from labeled data)
2. Splits into X_train_tuned, y_train_tuned and X_val, y_val
3. Keeps X_test as unlabeled data for final predictions

For Google Colab
"""

from sklearn.model_selection import train_test_split

print("=" * 80)
print("TRAIN/VALIDATION/TEST SPLIT")
print("=" * 80)

print("\n[Step 1] Splitting training data into train and validation...\n")

# Split training data into training and validation (80/20)
X_train_tuned, X_val, y_train_tuned, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"✓ Training set: {X_train_tuned.shape}")
print(f"  Churn rate: {y_train_tuned.mean()*100:.2f}%")
print(f"\n✓ Validation set: {X_val.shape}")
print(f"  Churn rate: {y_val.mean()*100:.2f}%")
print(f"\n✓ Test set (unlabeled): {X_test.shape}")

print(f"\n[Step 2] Verifying feature alignment...\n")

print(f"✓ All sets have same features: {list(X_train_tuned.columns) == list(X_val.columns) == list(X_test.columns)}")
print(f"✓ Feature count: {X_train_tuned.shape[1]}")

print("\n" + "=" * 80)
print("SPLIT COMPLETE")
print("=" * 80)
print(f"\nVariables available for modeling:")
print(f"  - X_train_tuned: {X_train_tuned.shape} - training data")
print(f"  - y_train_tuned: {y_train_tuned.shape} - training labels")
print(f"  - X_val: {X_val.shape} - validation data")
print(f"  - y_val: {y_val.shape} - validation labels")
print(f"  - X_test: {X_test.shape} - unlabeled test data for final predictions")
print(f"  - encoders: dict - for applying to new data")

print(f"\nNOTE: Do NOT try to evaluate against y_test (it doesn't exist)")
print(f"      Use X_val and y_val for hyperparameter tuning and validation")
print(f"      Use X_test for generating final submission predictions")
print("=" * 80)

TRAIN/VALIDATION/TEST SPLIT

[Step 1] Splitting training data into train and validation...

✓ Training set: (475355, 26)
  Churn rate: 22.52%

✓ Validation set: (118839, 26)
  Churn rate: 22.52%

✓ Test set (unlabeled): (254655, 26)

[Step 2] Verifying feature alignment...

✓ All sets have same features: True
✓ Feature count: 26

SPLIT COMPLETE

Variables available for modeling:
  - X_train_tuned: (475355, 26) - training data
  - y_train_tuned: (475355,) - training labels
  - X_val: (118839, 26) - validation data
  - y_val: (118839,) - validation labels
  - X_test: (254655, 26) - unlabeled test data for final predictions
  - encoders: dict - for applying to new data

NOTE: Do NOT try to evaluate against y_test (it doesn't exist)
      Use X_val and y_val for hyperparameter tuning and validation
      Use X_test for generating final submission predictions


In [ ]:
!pip install optuna

In [ ]:
"""
XGBOOST MODEL WITH HYPERPARAMETER TUNING
========================================

Separate Colab cell - copy and run after preprocessing pipeline

Includes:
- Optuna hyperparameter optimization
- Cross-validation
- Detailed evaluation metrics
- Feature importance visualization
"""

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.pruners import MedianPruner
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("XGBOOST MODEL WITH HYPERPARAMETER TUNING")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD PREPROCESSED DATA
# ============================================================================
print("\n[STEP 1] Loading preprocessed and split data...")

# Expected from preprocessing cell:
# X_train, y_train, encoders from train.csv
# X_test from test.csv (no labels)
# Then split training into train/val:
# X_train_tuned, X_val, y_train_tuned, y_val = train_test_split(...)

print(f"✓ Training set: {X_train_tuned.shape}")
print(f"✓ Validation set: {X_val.shape}")
print(f"✓ Test set (unlabeled): {X_test.shape}")
print(f"✓ Churn rate (train): {y_train_tuned.mean()*100:.2f}%")
print(f"✓ Churn rate (val): {y_val.mean()*100:.2f}%\n")

# ============================================================================
# STEP 2: DEFINE HYPERPARAMETER OPTIMIZATION OBJECTIVE
# ============================================================================
print("[STEP 2] Setting up Optuna hyperparameter optimization...\n")

def objective_xgb(trial):
    """
    Optuna objective function for XGBoost
    Uses validation set for evaluation
    """

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }

    # Train on training set, validate on validation set
    model = xgb.XGBClassifier(
        **params,
        use_label_encoder=False,
        eval_metric='logloss',
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )

    model.fit(
        X_train_tuned, y_train_tuned,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # Evaluate on validation set
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)

    # Report for pruning
    trial.report(auc, step=0)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return auc

# ============================================================================
# STEP 3: RUN OPTUNA OPTIMIZATION
# ============================================================================
print("[STEP 3] Running Optuna optimization (this may take a few minutes)...\n")

sampler = optuna.samplers.TPESampler(seed=42)
pruner = MedianPruner()

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner
)

study.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

# Get best parameters
best_params = study.best_params
best_auc_optuna = study.best_value

print(f"\n✓ Best AUC found: {best_auc_optuna:.4f}")
print(f"✓ Best parameters:")
for param, value in best_params.items():
    print(f"    {param}: {value}")

# ============================================================================
# STEP 4: RETRAIN MODEL ON ENTIRE TRAINING DATASET
# ============================================================================
print(f"\n[STEP 4] Retraining final XGBoost model on ENTIRE training set...\n")

# Use the entire X_train, y_train (not just train_tuned)
xgb_final = xgb.XGBClassifier(
    **best_params,
    use_label_encoder=False,
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42,
    verbosity=0
)

xgb_final.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],  # Still use val for monitoring, but train on full X_train
    verbose=False
)

print(f"✓ Model trained on entire training set: {X_train.shape}")

# ============================================================================
# STEP 5: EVALUATE ON VALIDATION SET
# ============================================================================
print(f"\n[STEP 5] Evaluating on validation set...\n")

y_pred_proba = xgb_final.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_val, y_pred_proba)
f1 = f1_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)

print(f"AUC-ROC: {auc:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(cm)
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['No Churn', 'Churn']))

# ============================================================================
# STEP 6: VISUALIZATIONS
# ============================================================================
print(f"\n[STEP 6] Creating visualizations...\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. ROC Curve
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
ax.plot(fpr, tpr, linewidth=3, label=f'XGBoost (AUC={auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax.set_title('ROC Curve (Validation Set)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Confusion Matrix
ax = axes[0, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
ax.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax.set_title('Confusion Matrix (Validation Set)', fontsize=12, fontweight='bold')
ax.set_xticklabels(['No Churn', 'Churn'])
ax.set_yticklabels(['No Churn', 'Churn'])

# 3. Feature Importance
ax = axes[1, 0]
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_final.feature_importances_
}).sort_values('importance', ascending=False).head(15)

ax.barh(range(len(feature_importance)), feature_importance['importance'], color='#e74c3c')
ax.set_yticks(range(len(feature_importance)))
ax.set_yticklabels(feature_importance['feature'], fontsize=9)
ax.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Features', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# 4. Optuna Trials History
ax = axes[1, 1]
trials_df = study.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], linewidth=2, marker='o', markersize=4)
ax.axhline(y=best_auc_optuna, color='r', linestyle='--', linewidth=2, label=f'Best: {best_auc_optuna:.4f}')
ax.set_xlabel('Trial Number', fontsize=11, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax.set_title('Optuna Optimization History', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('xgboost_results.png', dpi=150, bbox_inches='tight')
print("✓ Saved: xgboost_results.png\n")
plt.close()

# ============================================================================
# STEP 7: SAVE MODEL & RESULTS
# ============================================================================
print("[STEP 7] Saving model and results...\n")

# Save validation set predictions
xgb_val_predictions = pd.DataFrame({
    'actual': y_val.values,
    'predicted_proba': y_pred_proba,
    'predicted': y_pred
})
xgb_val_predictions.to_csv('xgboost_val_predictions.csv', index=False)
print("✓ Saved: xgboost_val_predictions.csv")

# Generate predictions on unlabeled test set
xgb_test_pred_proba = xgb_final.predict_proba(X_test)[:, 1]
xgb_test_pred_binary = (xgb_test_pred_proba > 0.5).astype(int)

# Create submission file with ACTUAL test IDs from the original test CSV
# Note: test_raw is the original test dataframe before preprocessing
submission_df = pd.DataFrame({
    'id': test_raw['id'].values,
    'churn': xgb_test_pred_binary
})
submission_df.to_csv('xgboost_submission.csv', index=False)
print("✓ Saved: xgboost_submission.csv (Kaggle submission format)")
print(f"  Shape: {submission_df.shape}")
print(f"  Sample:\n{submission_df.head()}")

# Also save probabilities for reference
xgboost_test_pred_df = pd.DataFrame({
    'id': test_raw['id'].values,
    'churn_probability': xgb_test_pred_proba
})
xgboost_test_pred_df.to_csv('xgboost_test_probabilities.csv', index=False)
print("✓ Saved: xgboost_test_probabilities.csv (for reference)")

# Save model
xgb_final.save_model('xgboost_model.json')
print("✓ Saved: xgboost_model.json")

# Save feature importance
feature_importance.to_csv('xgboost_feature_importance.csv', index=False)
print("✓ Saved: xgboost_feature_importance.csv")

# Save best parameters
import json
with open('xgboost_best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print("✓ Saved: xgboost_best_params.json")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("XGBOOST MODEL COMPLETE")
print("=" * 80)
print(f"\nBest Hyperparameters (from Optuna):")
for param, value in best_params.items():
    if not param.startswith('_'):
        print(f"  {param}: {value}")

print(f"\nTest Set Performance:")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  True Negatives: {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives: {cm[1, 1]}")

print(f"\nFiles Generated:")
print(f"  ✓ xgboost_results.png - Visualizations")
print(f"  ✓ xgboost_predictions.csv - Predictions on test set")
print(f"  ✓ xgboost_model.json - Trained model")
print(f"  ✓ xgboost_feature_importance.csv - Feature importance")
print(f"  ✓ xgboost_best_params.json - Best hyperparameters")

print(f"\nNext: Create cells for LightGBM, CatBoost, and Neural Network")
print("=" * 80)

[I 2026-03-21 14:29:45,118] A new study created in memory with name: no-name-7e454475-44c4-48e0-8700-b2384b0d7677


XGBOOST MODEL WITH HYPERPARAMETER TUNING

[STEP 1] Loading preprocessed and split data...
✓ Training set: (475355, 26)
✓ Validation set: (118839, 26)
✓ Test set (unlabeled): (254655, 26)
✓ Churn rate (train): 22.52%
✓ Churn rate (val): 22.52%

[STEP 2] Setting up Optuna hyperparameter optimization...

[STEP 3] Running Optuna optimization (this may take a few minutes)...



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-21 14:30:56,477] Trial 0 finished with value: 0.9158038831560555 and parameters: {'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'gamma': 0.2904180608409973, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.005575058716044}. Best is trial 0 with value: 0.9158038831560555.
[I 2026-03-21 14:31:48,212] Trial 1 finished with value: 0.9165702598778698 and parameters: {'n_estimators': 800, 'max_depth': 3, 'learning_rate': 0.2526878207508456, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'min_child_weight': 2, 'gamma': 0.9170225492671691, 'reg_alpha': 1.5212112147976886, 'reg_lambda': 2.6237821581611893}. Best is trial 1 with value: 0.9165702598778698.
[I 2026-03-21 14:32:34,678] Trial 2 finished with value: 0.9157350872235226 and parameters: {'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.032781876533976156, 'subsample': 0.

In [ ]:
"""
DATA SIZE DIAGNOSTIC
===================

Check if using full or sample datasets
"""

import pandas as pd

print("=" * 80)
print("DATA SIZE DIAGNOSTIC")
print("=" * 80)

# Check raw test file
try:
    test_raw_check = pd.read_csv('test.csv')
    print(f"\n✓ test.csv found")
    print(f"  Rows: {len(test_raw_check)}")
    print(f"  Columns: {len(test_raw_check.columns)}")
    print(f"  Sample IDs: {test_raw_check['id'].head().tolist()}")
except FileNotFoundError:
    print(f"\n✗ test.csv NOT found")

# Check raw train file
try:
    train_raw_check = pd.read_csv('train.csv')
    print(f"\n✓ train.csv found")
    print(f"  Rows: {len(train_raw_check)}")
    print(f"  Columns: {len(train_raw_check.columns)}")
except FileNotFoundError:
    print(f"\n✗ train.csv NOT found")

# Check current variables
print(f"\n✓ Current variables in memory:")
print(f"  X_train: {X_train.shape if 'X_train' in dir() else 'Not loaded'}")
print(f"  y_train: {y_train.shape if 'y_train' in dir() else 'Not loaded'}")
print(f"  X_test: {X_test.shape if 'X_test' in dir() else 'Not loaded'}")
print(f"  test_raw: {test_raw.shape if 'test_raw' in dir() else 'Not loaded'}")

print("\n" + "=" * 80)
print("IF test.csv has 254,655 rows: ✓ You have the FULL dataset")
print("IF test.csv has ~7,900 rows: ✗ You're using a SAMPLE")
print("=" * 80)

DATA SIZE DIAGNOSTIC

✓ test.csv found
  Rows: 254655
  Columns: 20
  Sample IDs: [594194, 594195, 594196, 594197, 594198]

✓ train.csv found
  Rows: 594194
  Columns: 21

✓ Current variables in memory:
  X_train: (7849, 26)
  y_train: (7849,)
  X_test: (7900, 26)
  test_raw: (7900, 20)

IF test.csv has 254,655 rows: ✓ You have the FULL dataset
IF test.csv has ~7,900 rows: ✗ You're using a SAMPLE


In [ ]:
"""
LIGHTGBM MODEL WITH HYPERPARAMETER TUNING
=========================================

Separate Colab cell - copy and run after XGBoost

Includes:
- Optuna hyperparameter optimization (50 trials)
- Validation set evaluation
- Feature importance visualization
- Submission file generation
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.pruners import MedianPruner
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("LIGHTGBM MODEL WITH HYPERPARAMETER TUNING")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD PREPROCESSED AND SPLIT DATA
# ============================================================================
print("\n[STEP 1] Loading preprocessed and split data...")

# Expected from previous cells:
# X_train_tuned, y_train_tuned (training set)
# X_val, y_val (validation set)
# X_test (unlabeled test set - no y_test!)
# test_raw (original test dataframe with IDs)

print(f"✓ Training set: {X_train_tuned.shape}")
print(f"✓ Validation set: {X_val.shape}")
print(f"✓ Test set (unlabeled): {X_test.shape}")
print(f"✓ Churn rate (train): {y_train_tuned.mean()*100:.2f}%")
print(f"✓ Churn rate (val): {y_val.mean()*100:.2f}%\n")

# ============================================================================
# STEP 2: DEFINE HYPERPARAMETER OPTIMIZATION OBJECTIVE
# ============================================================================
print("[STEP 2] Setting up Optuna hyperparameter optimization...\n")

def objective_lgb(trial):
    """
    Optuna objective function for LightGBM
    Uses validation set for evaluation
    """

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
    }

    # Train on training set, validate on validation set
    model = lgb.LGBMClassifier(
        **params,
        n_jobs=-1,
        verbose=-1,
        random_state=42
    )

    model.fit(
        X_train_tuned, y_train_tuned,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(20), lgb.log_evaluation(period=0)]
    )

    # Evaluate on validation set
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)

    # Report for pruning
    trial.report(auc, step=0)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return auc

# ============================================================================
# STEP 3: RUN OPTUNA OPTIMIZATION
# ============================================================================
print("[STEP 3] Running Optuna optimization (this may take a while)...\n")

sampler = optuna.samplers.TPESampler(seed=42)
pruner = MedianPruner()

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner
)

study.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

# Get best parameters
best_params = study.best_params
best_auc_optuna = study.best_value

print(f"\n✓ Best AUC found: {best_auc_optuna:.4f}")
print(f"✓ Best parameters:")
for param, value in best_params.items():
    print(f"    {param}: {value}")

# ============================================================================
# STEP 4: RETRAIN MODEL ON ENTIRE TRAINING DATASET
# ============================================================================
print(f"\n[STEP 4] Retraining final LightGBM model on ENTIRE training set...\n")

# Use the entire X_train, y_train (not just train_tuned)
lgb_final = lgb.LGBMClassifier(
    **best_params,
    n_jobs=-1,
    verbose=-1,
    random_state=42
)

lgb_final.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],  # Still use val for monitoring, but train on full X_train
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(period=0)]
)

print(f"✓ Model trained on entire training set: {X_train.shape}")

# ============================================================================
# STEP 5: EVALUATE ON VALIDATION SET
# ============================================================================
print(f"\n[STEP 5] Evaluating on validation set...\n")

y_pred_proba = lgb_final.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_val, y_pred_proba)
f1 = f1_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)

print(f"AUC-ROC: {auc:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(cm)
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['No Churn', 'Churn']))

# ============================================================================
# STEP 6: VISUALIZATIONS
# ============================================================================
print(f"\n[STEP 6] Creating visualizations...\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. ROC Curve
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
ax.plot(fpr, tpr, linewidth=3, label=f'LightGBM (AUC={auc:.4f})', color='#f39c12')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax.set_title('ROC Curve (Validation Set)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Confusion Matrix
ax = axes[0, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=ax, cbar=False)
ax.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax.set_title('Confusion Matrix (Validation Set)', fontsize=12, fontweight='bold')
ax.set_xticklabels(['No Churn', 'Churn'])
ax.set_yticklabels(['No Churn', 'Churn'])

# 3. Feature Importance
ax = axes[1, 0]
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_final.feature_importances_
}).sort_values('importance', ascending=False).head(15)

ax.barh(range(len(feature_importance)), feature_importance['importance'], color='#f39c12')
ax.set_yticks(range(len(feature_importance)))
ax.set_yticklabels(feature_importance['feature'], fontsize=9)
ax.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Features', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# 4. Optuna Trials History
ax = axes[1, 1]
trials_df = study.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], linewidth=2, marker='o', markersize=4, color='#f39c12')
ax.axhline(y=best_auc_optuna, color='r', linestyle='--', linewidth=2, label=f'Best: {best_auc_optuna:.4f}')
ax.set_xlabel('Trial Number', fontsize=11, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax.set_title('Optuna Optimization History', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lightgbm_results.png', dpi=150, bbox_inches='tight')
print("✓ Saved: lightgbm_results.png\n")
plt.close()

# ============================================================================
# STEP 7: SAVE MODEL & RESULTS
# ============================================================================
print("[STEP 7] Saving model and results...\n")

# Save validation set predictions
lgb_val_predictions = pd.DataFrame({
    'actual': y_val.values,
    'predicted_proba': y_pred_proba,
    'predicted': y_pred
})
lgb_val_predictions.to_csv('lightgbm_val_predictions.csv', index=False)
print("✓ Saved: lightgbm_val_predictions.csv")

# Generate predictions on unlabeled test set
lgb_test_pred_proba = lgb_final.predict_proba(X_test)[:, 1]
lgb_test_pred_binary = (lgb_test_pred_proba > 0.5).astype(int)

# Create submission file with ACTUAL test IDs from the original test CSV
submission_df = pd.DataFrame({
    'id': test_raw['id'].values,
    'churn': lgb_test_pred_binary
})
submission_df.to_csv('lightgbm_submission.csv', index=False)
print("✓ Saved: lightgbm_submission.csv (Kaggle submission format)")
print(f"  Shape: {submission_df.shape}")
print(f"  Sample:\n{submission_df.head()}")

# Also save probabilities for reference
lgb_test_pred_df = pd.DataFrame({
    'id': test_raw['id'].values,
    'churn_probability': lgb_test_pred_proba
})
lgb_test_pred_df.to_csv('lightgbm_test_probabilities.csv', index=False)
print("✓ Saved: lightgbm_test_probabilities.csv (for reference)")

# Save model
lgb_final.booster_.save_model('lightgbm_model.txt')
print("✓ Saved: lightgbm_model.txt")

# Save feature importance
feature_importance.to_csv('lightgbm_feature_importance.csv', index=False)
print("✓ Saved: lightgbm_feature_importance.csv")

# Save best parameters
import json
with open('lightgbm_best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print("✓ Saved: lightgbm_best_params.json")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("LIGHTGBM MODEL COMPLETE")
print("=" * 80)
print(f"\nBest Hyperparameters (from Optuna):")
for param, value in best_params.items():
    if not param.startswith('_'):
        print(f"  {param}: {value}")

print(f"\nValidation Set Performance:")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  True Negatives: {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives: {cm[1, 1]}")

print(f"\nFiles Generated:")
print(f"  ✓ lightgbm_submission.csv - KAGGLE SUBMISSION (id, churn) ⭐")
print(f"  ✓ lightgbm_results.png - Visualizations")
print(f"  ✓ lightgbm_val_predictions.csv - Predictions on validation set")
print(f"  ✓ lightgbm_test_probabilities.csv - Probabilities on test set")
print(f"  ✓ lightgbm_model.txt - Trained model (full X_train)")
print(f"  ✓ lightgbm_feature_importance.csv - Feature importance")
print(f"  ✓ lightgbm_best_params.json - Best hyperparameters")

print(f"\nTraining Strategy:")
print(f"  1. Optuna tuning: Used X_train_tuned (80%) and X_val (20%)")
print(f"  2. Final model: Retrained on ENTIRE X_train + y_train")
print(f"  3. Submission: Generated on unlabeled X_test")

print(f"\nNext: Create cell for CatBoost")
print("=" * 80)

[I 2026-03-21 16:26:23,205] A new study created in memory with name: no-name-01fdd41c-631c-4162-86fa-1d5359af8d4e


LIGHTGBM MODEL WITH HYPERPARAMETER TUNING

[STEP 1] Loading preprocessed and split data...
✓ Training set: (475355, 26)
✓ Validation set: (118839, 26)
✓ Test set (unlabeled): (254655, 26)
✓ Churn rate (train): 22.52%
✓ Churn rate (val): 22.52%

[STEP 2] Setting up Optuna hyperparameter optimization...

[STEP 3] Running Optuna optimization (this may take a while)...



  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[400]	valid_0's binary_logloss: 0.297798
[I 2026-03-21 16:27:20,682] Trial 0 finished with value: 0.916315814471778 and parameters: {'n_estimators': 400, 'max_depth': 10, 'num_leaves': 79, 'learning_rate': 0.030405325392865647, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'min_child_samples': 7, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.005575058716044}. Best is trial 0 with value: 0.916315814471778.
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[800]	valid_0's binary_logloss: 0.297487
[I 2026-03-21 16:28:06,892] Trial 1 finished with value: 0.916488467593472 and parameters: {'n_estimators': 800, 'max_depth': 3, 'num_leaves': 98, 'learning_rate': 0.11536162338241392, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'min_child_samples': 13, 'reg_alpha': 1.52121

In [ ]:
"""
CATBOOST MODEL WITH HYPERPARAMETER TUNING
=========================================

Separate Colab cell - copy and run after LightGBM

Includes:
- Optuna hyperparameter optimization (50 trials)
- Validation set evaluation
- Feature importance visualization
- Submission file generation
"""

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.pruners import MedianPruner
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("CATBOOST MODEL WITH HYPERPARAMETER TUNING")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD PREPROCESSED AND SPLIT DATA
# ============================================================================
print("\n[STEP 1] Loading preprocessed and split data...")

# Expected from previous cells:
# X_train_tuned, y_train_tuned (training set)
# X_val, y_val (validation set)
# X_test (unlabeled test set - no y_test!)
# test_raw (original test dataframe with IDs)

print(f"✓ Training set: {X_train_tuned.shape}")
print(f"✓ Validation set: {X_val.shape}")
print(f"✓ Test set (unlabeled): {X_test.shape}")
print(f"✓ Churn rate (train): {y_train_tuned.mean()*100:.2f}%")
print(f"✓ Churn rate (val): {y_val.mean()*100:.2f}%\n")

# ============================================================================
# STEP 2: DEFINE HYPERPARAMETER OPTIMIZATION OBJECTIVE
# ============================================================================
print("[STEP 2] Setting up Optuna hyperparameter optimization...\n")

def objective_cat(trial):
    """
    Optuna objective function for CatBoost
    Uses validation set for evaluation
    """

    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000, step=100),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0, 10),
        'random_strength': trial.suggest_float('random_strength', 0, 5),
    }

    # Train on training set, validate on validation set
    model = CatBoostClassifier(
        **params,
        verbose=0,
        random_state=42,
        thread_count=-1
    )

    model.fit(
        X_train_tuned, y_train_tuned,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=20,
        use_best_model=True
    )

    # Evaluate on validation set
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)

    # Report for pruning
    trial.report(auc, step=0)

    if trial.should_prune():
        raise optuna.TrialPruned()

    return auc

# ============================================================================
# STEP 3: RUN OPTUNA OPTIMIZATION
# ============================================================================
print("[STEP 3] Running Optuna optimization (this may take a while)...\n")

sampler = optuna.samplers.TPESampler(seed=42)
pruner = MedianPruner()

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner
)

study.optimize(objective_cat, n_trials=50, show_progress_bar=True)

# Get best parameters
best_params = study.best_params
best_auc_optuna = study.best_value

print(f"\n✓ Best AUC found: {best_auc_optuna:.4f}")
print(f"✓ Best parameters:")
for param, value in best_params.items():
    print(f"    {param}: {value}")

# ============================================================================
# STEP 4: RETRAIN MODEL ON ENTIRE TRAINING DATASET
# ============================================================================
print(f"\n[STEP 4] Retraining final CatBoost model on ENTIRE training set...\n")

# Use the entire X_train, y_train (not just train_tuned)
cat_final = CatBoostClassifier(
    **best_params,
    verbose=0,
    random_state=42,
    thread_count=-1
)

cat_final.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],  # Still use val for monitoring, but train on full X_train
    early_stopping_rounds=20,
    use_best_model=True
)

print(f"✓ Model trained on entire training set: {X_train.shape}")

# ============================================================================
# STEP 5: EVALUATE ON VALIDATION SET
# ============================================================================
print(f"\n[STEP 5] Evaluating on validation set...\n")

y_pred_proba = cat_final.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
auc = roc_auc_score(y_val, y_pred_proba)
f1 = f1_score(y_val, y_pred)
cm = confusion_matrix(y_val, y_pred)

print(f"AUC-ROC: {auc:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(cm)
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=['No Churn', 'Churn']))

# ============================================================================
# STEP 6: VISUALIZATIONS
# ============================================================================
print(f"\n[STEP 6] Creating visualizations...\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. ROC Curve
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
ax.plot(fpr, tpr, linewidth=3, label=f'CatBoost (AUC={auc:.4f})', color='#e74c3c')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
ax.set_title('ROC Curve (Validation Set)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 2. Confusion Matrix
ax = axes[0, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax, cbar=False)
ax.set_xlabel('Predicted', fontsize=11, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11, fontweight='bold')
ax.set_title('Confusion Matrix (Validation Set)', fontsize=12, fontweight='bold')
ax.set_xticklabels(['No Churn', 'Churn'])
ax.set_yticklabels(['No Churn', 'Churn'])

# 3. Feature Importance
ax = axes[1, 0]
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': cat_final.feature_importances_
}).sort_values('importance', ascending=False).head(15)

ax.barh(range(len(feature_importance)), feature_importance['importance'], color='#e74c3c')
ax.set_yticks(range(len(feature_importance)))
ax.set_yticklabels(feature_importance['feature'], fontsize=9)
ax.set_xlabel('Importance', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Features', fontsize=12, fontweight='bold')
ax.invert_yaxis()

# 4. Optuna Trials History
ax = axes[1, 1]
trials_df = study.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], linewidth=2, marker='o', markersize=4, color='#e74c3c')
ax.axhline(y=best_auc_optuna, color='r', linestyle='--', linewidth=2, label=f'Best: {best_auc_optuna:.4f}')
ax.set_xlabel('Trial Number', fontsize=11, fontweight='bold')
ax.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax.set_title('Optuna Optimization History', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('catboost_results.png', dpi=150, bbox_inches='tight')
print("✓ Saved: catboost_results.png\n")
plt.close()

# ============================================================================
# STEP 7: SAVE MODEL & RESULTS
# ============================================================================
print("[STEP 7] Saving model and results...\n")

# Save validation set predictions
cat_val_predictions = pd.DataFrame({
    'actual': y_val.values,
    'predicted_proba': y_pred_proba,
    'predicted': y_pred
})
cat_val_predictions.to_csv('catboost_val_predictions.csv', index=False)
print("✓ Saved: catboost_val_predictions.csv")

# Generate predictions on unlabeled test set
cat_test_pred_proba = cat_final.predict_proba(X_test)[:, 1]
cat_test_pred_binary = (cat_test_pred_proba > 0.5).astype(int)

# Create submission file with ACTUAL test IDs from the original test CSV
submission_df = pd.DataFrame({
    'id': test_raw['id'].values,
    'churn': cat_test_pred_binary
})
submission_df.to_csv('catboost_submission.csv', index=False)
print("✓ Saved: catboost_submission.csv (Kaggle submission format)")
print(f"  Shape: {submission_df.shape}")
print(f"  Sample:\n{submission_df.head()}")

# Also save probabilities for reference
cat_test_pred_df = pd.DataFrame({
    'id': test_raw['id'].values,
    'churn_probability': cat_test_pred_proba
})
cat_test_pred_df.to_csv('catboost_test_probabilities.csv', index=False)
print("✓ Saved: catboost_test_probabilities.csv (for reference)")

# Save model
cat_final.save_model('catboost_model.cbm')
print("✓ Saved: catboost_model.cbm")

# Save feature importance
feature_importance.to_csv('catboost_feature_importance.csv', index=False)
print("✓ Saved: catboost_feature_importance.csv")

# Save best parameters
import json
with open('catboost_best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print("✓ Saved: catboost_best_params.json")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("CATBOOST MODEL COMPLETE")
print("=" * 80)
print(f"\nBest Hyperparameters (from Optuna):")
for param, value in best_params.items():
    if not param.startswith('_'):
        print(f"  {param}: {value}")

print(f"\nValidation Set Performance:")
print(f"  AUC-ROC: {auc:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  True Negatives: {cm[0, 0]}")
print(f"  False Positives: {cm[0, 1]}")
print(f"  False Negatives: {cm[1, 0]}")
print(f"  True Positives: {cm[1, 1]}")

print(f"\nFiles Generated:")
print(f"  ✓ catboost_submission.csv - KAGGLE SUBMISSION (id, churn) ⭐")
print(f"  ✓ catboost_results.png - Visualizations")
print(f"  ✓ catboost_val_predictions.csv - Predictions on validation set")
print(f"  ✓ catboost_test_probabilities.csv - Probabilities on test set")
print(f"  ✓ catboost_model.cbm - Trained model (full X_train)")
print(f"  ✓ catboost_feature_importance.csv - Feature importance")
print(f"  ✓ catboost_best_params.json - Best hyperparameters")

print(f"\nTraining Strategy:")
print(f"  1. Optuna tuning: Used X_train_tuned (80%) and X_val (20%)")
print(f"  2. Final model: Retrained on ENTIRE X_train + y_train")
print(f"  3. Submission: Generated on unlabeled X_test")

print(f"\nNext: Create ensemble combining all 3 models")
print("=" * 80)

[I 2026-03-21 17:05:40,385] A new study created in memory with name: no-name-8170d641-e2f8-43cd-82dd-0b715cc3e507


CATBOOST MODEL WITH HYPERPARAMETER TUNING

[STEP 1] Loading preprocessed and split data...
✓ Training set: (475355, 26)
✓ Validation set: (118839, 26)
✓ Test set (unlabeled): (254655, 26)
✓ Churn rate (train): 22.52%
✓ Churn rate (val): 22.52%

[STEP 2] Setting up Optuna hyperparameter optimization...

[STEP 3] Running Optuna optimization (this may take a while)...



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-21 17:06:55,390] Trial 0 finished with value: 0.9159200042167492 and parameters: {'iterations': 400, 'depth': 10, 'learning_rate': 0.06504856968981275, 'subsample': 0.7993292420985183, 'colsample_bylevel': 0.5780093202212182, 'l2_leaf_reg': 1.5599452033620265, 'random_strength': 0.2904180608409973}. Best is trial 0 with value: 0.9159200042167492.
[I 2026-03-21 17:08:38,328] Trial 1 finished with value: 0.9166267661701712 and parameters: {'iterations': 900, 'depth': 7, 'learning_rate': 0.05675206026988748, 'subsample': 0.5102922471479012, 'colsample_bylevel': 0.9849549260809971, 'l2_leaf_reg': 8.324426408004218, 'random_strength': 1.0616955533913808}. Best is trial 1 with value: 0.9166267661701712.
[I 2026-03-21 17:08:57,972] Trial 2 finished with value: 0.9054480618145939 and parameters: {'iterations': 200, 'depth': 4, 'learning_rate': 0.005670807781371429, 'subsample': 0.762378215816119, 'colsample_bylevel': 0.7159725093210578, 'l2_leaf_reg': 2.9122914019804194, 'random_str

In [ ]:
"""
LOAD TRAINED MODELS
===================

Loads XGBoost, LightGBM, and CatBoost models from saved files
Run this FIRST before running the ensemble script
"""

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import pandas as pd
import numpy as np

print("=" * 80)
print("LOADING TRAINED MODELS")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD XGBOOST MODEL
# ============================================================================
print("\n[STEP 1] Loading XGBoost model...\n")

try:
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('xgboost_model.json')
    print("✓ XGBoost model loaded successfully")
except Exception as e:
    print(f"✗ Error loading XGBoost model: {e}")

# ============================================================================
# STEP 2: LOAD LIGHTGBM MODEL
# ============================================================================
print("\n[STEP 2] Loading LightGBM model...\n")

try:
    lgb_model = lgb.Booster(model_file='lightgbm_model.txt')
    print("✓ LightGBM model loaded successfully")
except Exception as e:
    print(f"✗ Error loading LightGBM model: {e}")

# ============================================================================
# STEP 3: LOAD CATBOOST MODEL
# ============================================================================
print("\n[STEP 3] Loading CatBoost model...\n")

try:
    cat_model = CatBoostClassifier()
    cat_model.load_model('catboost_model.cbm')
    print("✓ CatBoost model loaded successfully")
except Exception as e:
    print(f"✗ Error loading CatBoost model: {e}")

# ============================================================================
# STEP 4: VERIFY MODELS ARE READY
# ============================================================================
print("\n" + "=" * 80)
print("MODEL VERIFICATION")
print("=" * 80)

print("\n✓ All 3 models are ready to use!")
print("\nYou can now:")
print("  1. Make predictions with: xgb_model.predict_proba(X)")
print("  2. Make predictions with: lgb_model.predict(X)")
print("  3. Make predictions with: cat_model.predict_proba(X)")
print("\nOr run the ensemble script next!")
print("\n" + "=" * 80)

LOADING TRAINED MODELS

[STEP 1] Loading XGBoost model...

✓ XGBoost model loaded successfully

[STEP 2] Loading LightGBM model...

✓ LightGBM model loaded successfully

[STEP 3] Loading CatBoost model...

✓ CatBoost model loaded successfully

MODEL VERIFICATION

✓ All 3 models are ready to use!

You can now:
  1. Make predictions with: xgb_model.predict_proba(X)
  2. Make predictions with: lgb_model.predict(X)
  3. Make predictions with: cat_model.predict_proba(X)

Or run the ensemble script next!



In [ ]:
"""
LOAD MODELS & REGENERATE PREDICTIONS
===================================

Loads trained models and generates all prediction files needed for ensemble
Run this FIRST before running the ensemble script
"""

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("LOAD MODELS & REGENERATE PREDICTIONS")
print("=" * 80)

# ============================================================================
# STEP 0: LOAD PREPROCESSED DATA
# ============================================================================
print("\n[STEP 0] Loading preprocessed data...\n")

# Load train and test
train_raw = pd.read_csv('train.csv')
test_raw = pd.read_csv('test.csv')

print(f"✓ train.csv: {len(train_raw)} rows")
print(f"✓ test.csv: {len(test_raw)} rows")

# Load preprocessing function
print("\nℹ️  Make sure preprocessing_pipeline function is defined in previous cell")
print("    Or load the preprocessing pipeline file")

# Run preprocessing
print("\n[Preprocessing] Running preprocessing on train and test data...\n")

X_train, y_train, encoders = preprocess_pipeline(train_raw, fit_encoders=True)
X_test, _, _ = preprocess_pipeline(test_raw, fit_encoders=False, encoders_dict=encoders)

print(f"✓ X_train: {X_train.shape}")
print(f"✓ y_train: {y_train.shape}")
print(f"✓ X_test: {X_test.shape}")

# Create train/val split (same as before)
X_train_tuned, X_val, y_train_tuned, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"✓ X_train_tuned: {X_train_tuned.shape}")
print(f"✓ X_val: {X_val.shape}\n")

# ============================================================================
# STEP 1: LOAD XGBOOST MODEL
# ============================================================================
print("[STEP 1] Loading XGBoost model...\n")

try:
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('xgboost_model.json')
    print("✓ XGBoost model loaded\n")

    # Generate predictions
    print("  Generating XGBoost predictions...")
    xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]
    xgb_val_pred = (xgb_val_proba > 0.5).astype(int)
    xgb_test_proba = xgb_model.predict_proba(X_test)[:, 1]

    # Save validation predictions
    xgb_val_df = pd.DataFrame({
        'actual': y_val.values,
        'predicted_proba': xgb_val_proba,
        'predicted': xgb_val_pred
    })
    xgb_val_df.to_csv('xgboost_val_predictions.csv', index=False)
    print("  ✓ Saved: xgboost_val_predictions.csv")

    # Save test probabilities
    xgb_test_df = pd.DataFrame({
        'id': test_raw['id'].values,
        'churn_probability': xgb_test_proba
    })
    xgb_test_df.to_csv('xgboost_test_probabilities.csv', index=False)
    print("  ✓ Saved: xgboost_test_probabilities.csv\n")

except Exception as e:
    print(f"✗ Error with XGBoost: {e}\n")

# ============================================================================
# STEP 2: LOAD LIGHTGBM MODEL
# ============================================================================
print("[STEP 2] Loading LightGBM model...\n")

try:
    lgb_model = lgb.Booster(model_file='lightgbm_model.txt')
    print("✓ LightGBM model loaded\n")

    # Generate predictions
    print("  Generating LightGBM predictions...")
    lgb_val_proba = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
    lgb_val_pred = (lgb_val_proba > 0.5).astype(int)
    lgb_test_proba = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)

    # Save validation predictions
    lgb_val_df = pd.DataFrame({
        'actual': y_val.values,
        'predicted_proba': lgb_val_proba,
        'predicted': lgb_val_pred
    })
    lgb_val_df.to_csv('lightgbm_val_predictions.csv', index=False)
    print("  ✓ Saved: lightgbm_val_predictions.csv")

    # Save test probabilities
    lgb_test_df = pd.DataFrame({
        'id': test_raw['id'].values,
        'churn_probability': lgb_test_proba
    })
    lgb_test_df.to_csv('lightgbm_test_probabilities.csv', index=False)
    print("  ✓ Saved: lightgbm_test_probabilities.csv\n")

except Exception as e:
    print(f"✗ Error with LightGBM: {e}\n")

# ============================================================================
# STEP 3: LOAD CATBOOST MODEL
# ============================================================================
print("[STEP 3] Loading CatBoost model...\n")

try:
    cat_model = CatBoostClassifier()
    cat_model.load_model('catboost_model.cbm')
    print("✓ CatBoost model loaded\n")

    # Generate predictions
    print("  Generating CatBoost predictions...")
    cat_val_proba = cat_model.predict_proba(X_val)[:, 1]
    cat_val_pred = (cat_val_proba > 0.5).astype(int)
    cat_test_proba = cat_model.predict_proba(X_test)[:, 1]

    # Save validation predictions
    cat_val_df = pd.DataFrame({
        'actual': y_val.values,
        'predicted_proba': cat_val_proba,
        'predicted': cat_val_pred
    })
    cat_val_df.to_csv('catboost_val_predictions.csv', index=False)
    print("  ✓ Saved: catboost_val_predictions.csv")

    # Save test probabilities
    cat_test_df = pd.DataFrame({
        'id': test_raw['id'].values,
        'churn_probability': cat_test_proba
    })
    cat_test_df.to_csv('catboost_test_probabilities.csv', index=False)
    print("  ✓ Saved: catboost_test_probabilities.csv\n")

except Exception as e:
    print(f"✗ Error with CatBoost: {e}\n")

# ============================================================================
# STEP 4: CREATE SUBMISSION FILE WITH TEST IDS
# ============================================================================
print("[STEP 4] Creating submission template...\n")

try:
    xgb_submission = pd.DataFrame({
        'id': test_raw['id'].values,
        'churn': (xgb_test_proba > 0.5).astype(int)
    })
    xgb_submission.to_csv('xgboost_submission.csv', index=False)
    print("✓ Saved: xgboost_submission.csv\n")
except Exception as e:
    print(f"✗ Error creating submission: {e}\n")

# ============================================================================
# STEP 5: SUMMARY
# ============================================================================
print("=" * 80)
print("MODELS LOADED & PREDICTIONS REGENERATED")
print("=" * 80)

print("\n✓ Files created:")
print("  • xgboost_val_predictions.csv")
print("  • xgboost_test_probabilities.csv")
print("  • lightgbm_val_predictions.csv")
print("  • lightgbm_test_probabilities.csv")
print("  • catboost_val_predictions.csv")
print("  • catboost_test_probabilities.csv")
print("  • xgboost_submission.csv")

print("\n✓ Next step: Run the ensemble script!")
print("  Copy and run: 06_advanced_ensemble.py")

print("\n" + "=" * 80)

LOAD MODELS & REGENERATE PREDICTIONS

[STEP 0] Loading preprocessed data...

✓ train.csv: 594194 rows
✓ test.csv: 254655 rows

ℹ️  Make sure preprocessing_pipeline function is defined in previous cell
    Or load the preprocessing pipeline file

[Preprocessing] Running preprocessing on train and test data...

PREPROCESSING PIPELINE

✓ Extracted target variable
  Shape: (594194, 19)

✓ Numeric features: 4
✓ Categorical features: 15

[STEP 1] Engineering numeric features...
✓ Engineered numeric features: ['tenure_log', 'MonthlyCharges_log', 'charge_per_month', 'TotalCharges_log', 'charge_ratio']

[STEP 2] Encoding categorical features...
  Mode: FITTING new encoders
    ✓ Target encoded: Contract
    ✓ Target encoded: PaymentMethod
    ✓ Target encoded: InternetService
    ✓ One-hot encoded: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

[STEP 

In [ ]:
"""
ADVANCED ENSEMBLE WITH THRESHOLD OPTIMIZATION
==============================================

Tests multiple ensemble strategies with threshold tuning on validation set
Generates best submissions ready for Kaggle
"""

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 80)
print("ADVANCED ENSEMBLE WITH THRESHOLD OPTIMIZATION")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD ALL PREDICTIONS
# ============================================================================
print("\n[STEP 1] Loading predictions from all 3 models...\n")

# Validation predictions
xgb_val_df = pd.read_csv('xgboost_val_predictions.csv')
lgb_val_df = pd.read_csv('lightgbm_val_predictions.csv')
cat_val_df = pd.read_csv('catboost_val_predictions.csv')

y_val_true = xgb_val_df['actual'].values
xgb_val_proba = xgb_val_df['predicted_proba'].values
lgb_val_proba = lgb_val_df['predicted_proba'].values
cat_val_proba = cat_val_df['predicted_proba'].values

print(f"✓ Validation set size: {len(y_val_true)}")
print(f"✓ Churn rate: {y_val_true.mean()*100:.2f}%")

# Test predictions
xgb_test_proba = pd.read_csv('xgboost_test_probabilities.csv')['churn_probability'].values
lgb_test_proba = pd.read_csv('lightgbm_test_probabilities.csv')['churn_probability'].values
cat_test_proba = pd.read_csv('catboost_test_probabilities.csv')['churn_probability'].values
test_ids = pd.read_csv('xgboost_submission.csv')['id'].values

print(f"✓ Test set size: {len(test_ids)}\n")

# ============================================================================
# STEP 2: CREATE ENSEMBLE STRATEGIES
# ============================================================================
print("[STEP 2] Creating ensemble strategies...\n")

# Strategy 1: Simple Average
ensemble_avg = (xgb_val_proba + lgb_val_proba + cat_val_proba) / 3
ensemble_avg_test = (xgb_test_proba + lgb_test_proba + cat_test_proba) / 3

# Strategy 2: Weighted Average (LightGBM best, so higher weight)
weights = np.array([0.30, 0.40, 0.30])  # XGB, LGB, CAT
ensemble_weighted = (xgb_val_proba * weights[0] +
                     lgb_val_proba * weights[1] +
                     cat_val_proba * weights[2])
ensemble_weighted_test = (xgb_test_proba * weights[0] +
                          lgb_test_proba * weights[1] +
                          cat_test_proba * weights[2])

# Strategy 3: Median (robust to outliers)
ensemble_median = np.median([xgb_val_proba, lgb_val_proba, cat_val_proba], axis=0)
ensemble_median_test = np.median([xgb_test_proba, lgb_test_proba, cat_test_proba], axis=0)

# Strategy 4: Geometric Mean (often better than arithmetic)
ensemble_geom = (xgb_val_proba * lgb_val_proba * cat_val_proba) ** (1/3)
ensemble_geom_test = (xgb_test_proba * lgb_test_proba * cat_test_proba) ** (1/3)

# Strategy 5: Rank-based (rank each model's predictions, then average ranks)
xgb_rank = pd.Series(xgb_val_proba).rank().values / len(xgb_val_proba)
lgb_rank = pd.Series(lgb_val_proba).rank().values / len(lgb_val_proba)
cat_rank = pd.Series(cat_val_proba).rank().values / len(cat_val_proba)
ensemble_rank = (xgb_rank + lgb_rank + cat_rank) / 3

xgb_rank_test = pd.Series(xgb_test_proba).rank().values / len(xgb_test_proba)
lgb_rank_test = pd.Series(lgb_test_proba).rank().values / len(lgb_test_proba)
cat_rank_test = pd.Series(cat_test_proba).rank().values / len(cat_test_proba)
ensemble_rank_test = (xgb_rank_test + lgb_rank_test + cat_rank_test) / 3

print("✓ Simple Average")
print("✓ Weighted Average (XGB:30%, LGB:40%, CAT:30%)")
print("✓ Median")
print("✓ Geometric Mean")
print("✓ Rank-based\n")

# ============================================================================
# STEP 3: FIND OPTIMAL THRESHOLD FOR EACH STRATEGY
# ============================================================================
print("[STEP 3] Finding optimal thresholds on validation set...\n")

ensembles = {
    'average': (ensemble_avg, ensemble_avg_test),
    'weighted': (ensemble_weighted, ensemble_weighted_test),
    'median': (ensemble_median, ensemble_median_test),
    'geometric': (ensemble_geom, ensemble_geom_test),
    'rank': (ensemble_rank, ensemble_rank_test),
}

results = {}
best_thresholds = {}

for name, (val_proba, test_proba) in ensembles.items():
    print(f"\n{name.upper()}:")
    print("-" * 60)

    best_f1 = 0
    best_threshold = 0.5
    threshold_results = []

    # Test thresholds from 0.3 to 0.7
    for threshold in np.arange(0.3, 0.71, 0.01):
        pred = (val_proba > threshold).astype(int)

        f1 = f1_score(y_val_true, pred)
        precision = precision_score(y_val_true, pred)
        recall = recall_score(y_val_true, pred)
        auc = roc_auc_score(y_val_true, val_proba)

        threshold_results.append({
            'threshold': threshold,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'auc': auc
        })

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    # Show top 5 thresholds
    sorted_results = sorted(threshold_results, key=lambda x: x['f1'], reverse=True)
    print(f"{'Threshold':<12} {'F1':<10} {'Precision':<12} {'Recall':<10} {'AUC':<10}")
    for i, res in enumerate(sorted_results[:5]):
        marker = "✓ BEST" if i == 0 else ""
        print(f"{res['threshold']:<12.2f} {res['f1']:<10.4f} {res['precision']:<12.4f} {res['recall']:<10.4f} {res['auc']:<10.4f} {marker}")

    results[name] = {
        'best_f1': best_f1,
        'best_threshold': best_threshold,
        'all_results': sorted_results
    }
    best_thresholds[name] = best_threshold

# ============================================================================
# STEP 4: GENERATE SUBMISSIONS WITH OPTIMAL THRESHOLDS
# ============================================================================
print("\n\n[STEP 4] Generating submissions with optimal thresholds...\n")

submissions = {}

for name, (val_proba, test_proba) in ensembles.items():
    threshold = best_thresholds[name]
    test_pred = (test_proba > threshold).astype(int)

    submission = pd.DataFrame({
        'id': test_ids,
        'churn': test_pred
    })

    filename = f'ensemble_{name}_submission.csv'
    submission.to_csv(filename, index=False)
    submissions[name] = submission

    churn_count = test_pred.sum()
    print(f"✓ Saved: {filename}")
    print(f"  Threshold: {threshold:.2f}, Churn predictions: {churn_count:,} / {len(test_ids):,}")

# ============================================================================
# STEP 5: COMPARISON VISUALIZATION
# ============================================================================
print("\n\n[STEP 5] Creating comparison visualizations...\n")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Ensemble Strategy Comparison on Validation Set', fontsize=14, fontweight='bold')

ensemble_names = list(ensembles.keys())
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for idx, (name, color) in enumerate(zip(ensemble_names, colors)):
    ax = axes[idx // 3, idx % 3]

    all_results = results[name]['all_results']
    thresholds = [r['threshold'] for r in all_results]
    f1_scores = [r['f1'] for r in all_results]
    precisions = [r['precision'] for r in all_results]
    recalls = [r['recall'] for r in all_results]

    ax.plot(thresholds, f1_scores, marker='o', label='F1', linewidth=2, color=color)
    ax.plot(thresholds, precisions, marker='s', label='Precision', linewidth=2, alpha=0.7)
    ax.plot(thresholds, recalls, marker='^', label='Recall', linewidth=2, alpha=0.7)

    best_th = best_thresholds[name]
    best_f1 = results[name]['best_f1']
    ax.axvline(x=best_th, color='red', linestyle='--', alpha=0.5, label=f'Optimal: {best_th:.2f}')

    ax.set_xlabel('Threshold', fontweight='bold')
    ax.set_ylabel('Score', fontweight='bold')
    ax.set_title(f'{name.upper()}\nBest F1: {best_f1:.4f}', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.6, 0.75])

# Remove the last empty subplot
axes[1, 2].remove()

plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Saved: ensemble_comparison.png\n")
plt.close()

# ============================================================================
# STEP 6: SUMMARY & RECOMMENDATIONS
# ============================================================================
print("\n" + "=" * 80)
print("ENSEMBLE OPTIMIZATION COMPLETE")
print("=" * 80)

print("\n📊 SUMMARY TABLE:")
print("-" * 80)
print(f"{'Strategy':<15} {'Best F1':<12} {'Best Threshold':<18} {'Churn Count':<12}")
print("-" * 80)

for name, submission in submissions.items():
    f1 = results[name]['best_f1']
    threshold = best_thresholds[name]
    churn_count = submission['churn'].sum()
    print(f"{name:<15} {f1:<12.4f} {threshold:<18.2f} {churn_count:<12,}")

print("\n🎯 RECOMMENDATIONS:")
print("-" * 80)

# Find best strategy
best_strategy = max(results.items(), key=lambda x: x[1]['best_f1'])
print(f"\n1. BEST OVERALL: {best_strategy[0].upper()}")
print(f"   F1 Score: {best_strategy[1]['best_f1']:.4f}")
print(f"   Threshold: {best_thresholds[best_strategy[0]]:.2f}")
print(f"   → Submit: ensemble_{best_strategy[0]}_submission.csv")

print(f"\n2. ALTERNATIVES (try these if first doesn't work):")
sorted_by_f1 = sorted(results.items(), key=lambda x: x[1]['best_f1'], reverse=True)
for i, (name, result) in enumerate(sorted_by_f1[1:4], 2):
    print(f"   {i}. {name.upper()}: F1={result['best_f1']:.4f}, Threshold={best_thresholds[name]:.2f}")

print("\n" + "=" * 80)
print("FILES READY FOR KAGGLE SUBMISSION:")
print("=" * 80)
print("\n✓ ensemble_average_submission.csv")
print("✓ ensemble_weighted_submission.csv")
print("✓ ensemble_median_submission.csv")
print("✓ ensemble_geometric_submission.csv")
print("✓ ensemble_rank_submission.csv")
print("\n→ Start with: ensemble_{}.csv".format(best_strategy[0]))
print("=" * 80)

ADVANCED ENSEMBLE WITH THRESHOLD OPTIMIZATION

[STEP 1] Loading predictions from all 3 models...

✓ Validation set size: 118839
✓ Churn rate: 22.52%
✓ Test set size: 254655

[STEP 2] Creating ensemble strategies...

✓ Simple Average
✓ Weighted Average (XGB:30%, LGB:40%, CAT:30%)
✓ Median
✓ Geometric Mean
✓ Rank-based

[STEP 3] Finding optimal thresholds on validation set...


AVERAGE:
------------------------------------------------------------
Threshold    F1         Precision    Recall     AUC       
0.37         0.7079     0.6478       0.7802     0.9200     ✓ BEST
0.36         0.7078     0.6424       0.7881     0.9200     
0.38         0.7076     0.6536       0.7714     0.9200     
0.34         0.7073     0.6310       0.8046     0.9200     
0.35         0.7072     0.6364       0.7957     0.9200     

WEIGHTED:
------------------------------------------------------------
Threshold    F1         Precision    Recall     AUC       
0.37         0.7079     0.6479       0.7803     0.9201 

In [ ]:
"""
THRESHOLD A/B TESTING
====================

Creates submission files with multiple thresholds to test on Kaggle
Run after ensemble script
"""

import pandas as pd
import numpy as np

print("=" * 80)
print("THRESHOLD A/B TESTING")
print("=" * 80)

# Load test probabilities
print("\n[STEP 1] Loading ensemble probabilities...\n")

xgb_test_proba = pd.read_csv('xgboost_test_probabilities.csv')['churn_probability'].values
lgb_test_proba = pd.read_csv('lightgbm_test_probabilities.csv')['churn_probability'].values
cat_test_proba = pd.read_csv('catboost_test_probabilities.csv')['churn_probability'].values
test_ids = pd.read_csv('xgboost_submission.csv')['id'].values

# Create weighted ensemble
weights = np.array([0.30, 0.40, 0.30])  # XGB, LGB, CAT
ensemble_weighted = (xgb_test_proba * weights[0] +
                     lgb_test_proba * weights[1] +
                     cat_test_proba * weights[2])

print(f"✓ Ensemble probabilities loaded: {len(ensemble_weighted)}")
print(f"✓ Test IDs: {len(test_ids)}\n")

# ============================================================================
# STEP 2: CREATE SUBMISSIONS WITH DIFFERENT THRESHOLDS
# ============================================================================
print("[STEP 2] Creating submissions with different thresholds...\n")

# Test these thresholds
thresholds_to_test = [0.30, 0.35, 0.37, 0.40, 0.45, 0.47, 0.50, 0.55]

submissions = {}

print(f"{'Threshold':<12} {'Churn Count':<15} {'% Churn':<10} {'File':<45}")
print("-" * 82)

for threshold in thresholds_to_test:
    test_pred = (ensemble_weighted > threshold).astype(int)
    churn_count = test_pred.sum()
    churn_pct = (churn_count / len(test_ids)) * 100

    submission = pd.DataFrame({
        'id': test_ids,
        'churn': test_pred
    })

    filename = f'ensemble_weighted_threshold_{threshold:.2f}.csv'
    submission.to_csv(filename, index=False)
    submissions[threshold] = (churn_count, churn_pct)

    # Highlight key thresholds
    marker = ""
    if threshold == 0.37:
        marker = "  ← OPTIMAL (F1)"
    elif threshold == 0.50:
        marker = "  ← DEFAULT"
    elif threshold == 0.47:
        marker = "  ← HIGH PRECISION"

    print(f"{threshold:<12.2f} {churn_count:<15,} {churn_pct:<10.2f} {filename:<45} {marker}")

print("\n" + "=" * 80)
print("TESTING STRATEGY")
print("=" * 80)

print("\n1️⃣  FIRST: Submit threshold 0.37 (best F1)")
print("   File: ensemble_weighted_threshold_0.37.csv")

print("\n2️⃣  SECOND: If score is lower than expected, try 0.47")
print("   File: ensemble_weighted_threshold_0.47.csv")
print("   (Higher precision, fewer false positives)")

print("\n3️⃣  THIRD: If still not great, try 0.45")
print("   File: ensemble_weighted_threshold_0.45.csv")
print("   (Middle ground)")

print("\n💡 WHY DIFFERENT THRESHOLDS?")
print("   - 0.37: Best F1 on validation → likely best overall")
print("   - 0.47: Best precision → fewer false alarms")
print("   - 0.50: Default threshold → baseline")
print("   - 0.55+: More conservative → catches obvious churners")

print("\n📊 CHURN DISTRIBUTION:")
for threshold in sorted(submissions.keys()):
    churn_count, churn_pct = submissions[threshold]
    print(f"   {threshold:.2f}: {churn_count:,} / {len(test_ids):,} ({churn_pct:.2f}%)")

print("\n" + "=" * 80)
print(f"\n✓ Total files created: {len(submissions)}")
print("✓ Try them on Kaggle and pick the best!")
print("\n" + "=" * 80)

THRESHOLD A/B TESTING

[STEP 1] Loading ensemble probabilities...

✓ Ensemble probabilities loaded: 254655
✓ Test IDs: 254655

[STEP 2] Creating submissions with different thresholds...

Threshold    Churn Count     % Churn    File                                         
----------------------------------------------------------------------------------
0.30         76,154          29.90      ensemble_weighted_threshold_0.30.csv          
0.35         69,285          27.21      ensemble_weighted_threshold_0.35.csv          
0.37         66,582          26.15      ensemble_weighted_threshold_0.37.csv            ← OPTIMAL (F1)
0.40         62,555          24.56      ensemble_weighted_threshold_0.40.csv          
0.45         56,083          22.02      ensemble_weighted_threshold_0.45.csv          
0.47         53,391          20.97      ensemble_weighted_threshold_0.47.csv            ← HIGH PRECISION
0.50         49,352          19.38      ensemble_weighted_threshold_0.50.csv            

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

print("=" * 80)
print("ENSEMBLE WEIGHT & THRESHOLD OPTIMIZATION")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD VALIDATION PREDICTIONS
# ============================================================================
print("\n[STEP 1] Loading validation predictions...\n")

xgb_val_df = pd.read_csv('xgboost_val_predictions.csv')
lgb_val_df = pd.read_csv('lightgbm_val_predictions.csv')
cat_val_df = pd.read_csv('catboost_val_predictions.csv')

y_val_true = xgb_val_df['actual'].values
xgb_val_proba = xgb_val_df['predicted_proba'].values
lgb_val_proba = lgb_val_df['predicted_proba'].values
cat_val_proba = cat_val_df['predicted_proba'].values

print(f"✓ Validation set size: {len(y_val_true)}")
print(f"✓ Churn rate: {y_val_true.mean()*100:.2f}%\n")

# Load test probabilities
xgb_test_proba = pd.read_csv('xgboost_test_probabilities.csv')['churn_probability'].values
lgb_test_proba = pd.read_csv('lightgbm_test_probabilities.csv')['churn_probability'].values
cat_test_proba = pd.read_csv('catboost_test_probabilities.csv')['churn_probability'].values
test_ids = pd.read_csv('xgboost_submission.csv')['id'].values

print(f"✓ Test set size: {len(test_ids)}\n")

# ============================================================================
# STEP 2: GRID SEARCH FOR OPTIMAL WEIGHTS & THRESHOLD
# ============================================================================
print("[STEP 2] Grid search for optimal weights & threshold...\n")
print("This may take a minute or two...\n")

# Define weight ranges (must sum to 1)
# Test: 0.2, 0.25, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5
weight_options = np.arange(20, 51, 5) / 100  # 0.20, 0.25, 0.30, ..., 0.50
thresholds = np.arange(0.30, 0.56, 0.01)  # 0.30 to 0.55

results = []
best_f1 = 0
best_config = None

total_configs = 0
for xgb_w in weight_options:
    for lgb_w in weight_options:
        cat_w = 1.0 - xgb_w - lgb_w
        if cat_w < 0 or cat_w > 1:
            continue
        total_configs += 1

print(f"Testing {total_configs} weight combinations × {len(thresholds)} thresholds = {total_configs * len(thresholds):,} total configs\n")

config_count = 0
for xgb_w in weight_options:
    for lgb_w in weight_options:
        cat_w = 1.0 - xgb_w - lgb_w
        if cat_w < 0 or cat_w > 1:
            continue

        # Create ensemble with these weights
        ensemble_val = (xgb_val_proba * xgb_w +
                       lgb_val_proba * lgb_w +
                       cat_val_proba * cat_w)

        # Test different thresholds
        for threshold in thresholds:
            pred = (ensemble_val > threshold).astype(int)
            f1 = f1_score(y_val_true, pred)
            auc = roc_auc_score(y_val_true, ensemble_val)

            results.append({
                'xgb_weight': round(xgb_w, 2),
                'lgb_weight': round(lgb_w, 2),
                'cat_weight': round(cat_w, 2),
                'threshold': round(threshold, 2),
                'f1': f1,
                'auc': auc
            })

            if f1 > best_f1:
                best_f1 = f1
                best_config = {
                    'xgb_weight': round(xgb_w, 2),
                    'lgb_weight': round(lgb_w, 2),
                    'cat_weight': round(cat_w, 2),
                    'threshold': round(threshold, 2),
                    'f1': f1,
                    'auc': auc
                }

        config_count += 1
        if config_count % max(1, total_configs // 10) == 0:
            print(f"  Progress: {config_count}/{total_configs} weight configs tested...")

print(f"\n✓ Grid search complete!\n")

# ============================================================================
# STEP 3: ANALYZE RESULTS
# ============================================================================
print("[STEP 3] Analyzing results...\n")

results_df = pd.DataFrame(results)
best_results = results_df.nlargest(10, 'f1')

print("TOP 10 CONFIGURATIONS:")
print("-" * 100)
print(f"{'XGB':<8} {'LGB':<8} {'CAT':<8} {'Threshold':<12} {'F1':<10} {'AUC':<10}")
print("-" * 100)

for idx, row in best_results.iterrows():
    marker = " ✓ BEST" if row['f1'] == best_f1 else ""
    print(f"{row['xgb_weight']:<8.2f} {row['lgb_weight']:<8.2f} {row['cat_weight']:<8.2f} "
          f"{row['threshold']:<12.2f} {row['f1']:<10.4f} {row['auc']:<10.4f}{marker}")

# ============================================================================
# STEP 4: APPLY BEST CONFIG TO TEST SET
# ============================================================================
print(f"\n[STEP 4] Applying best configuration to test set...\n")

best_xgb_w = best_config['xgb_weight']
best_lgb_w = best_config['lgb_weight']
best_cat_w = best_config['cat_weight']
best_threshold = best_config['threshold']

print(f"Best weights found:")
print(f"  XGBoost: {best_xgb_w:.2f} ({best_xgb_w*100:.0f}%)")
print(f"  LightGBM: {best_lgb_w:.2f} ({best_lgb_w*100:.0f}%)")
print(f"  CatBoost: {best_cat_w:.2f} ({best_cat_w*100:.0f}%)")
print(f"  Threshold: {best_threshold:.2f}")
print(f"  Validation F1: {best_config['f1']:.4f}")
print(f"  Validation AUC: {best_config['auc']:.4f}\n")

# Create ensemble with best weights
ensemble_test = (xgb_test_proba * best_xgb_w +
                 lgb_test_proba * best_lgb_w +
                 cat_test_proba * best_cat_w)

# Apply best threshold
test_pred = (ensemble_test > best_threshold).astype(int)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'churn': test_pred
})

filename = f'ensemble_optimized_xgb{best_xgb_w:.2f}_lgb{best_lgb_w:.2f}_cat{best_cat_w:.2f}_th{best_threshold:.2f}.csv'
submission.to_csv(filename, index=False)

churn_count = test_pred.sum()
print(f"✓ Saved: {filename}")
print(f"  Churn predictions: {churn_count:,} / {len(test_ids):,}")
print(f"  Churn rate: {(churn_count/len(test_ids))*100:.2f}%\n")

# ============================================================================
# STEP 5: COMPARE WITH PREVIOUS BEST
# ============================================================================
print("[STEP 5] Comparison with previous best...\n")

# Previous best
prev_weights = (0.30, 0.40, 0.30)
prev_threshold = 0.37

prev_ensemble = (xgb_val_proba * prev_weights[0] +
                 lgb_val_proba * prev_weights[1] +
                 cat_val_proba * prev_weights[2])
prev_pred = (prev_ensemble > prev_threshold).astype(int)
prev_f1 = f1_score(y_val_true, prev_pred)

print(f"{'Model':<30} {'Weights':<30} {'Threshold':<12} {'F1':<10}")
print("-" * 82)
print(f"{'Previous (weighted)':<30} {str(prev_weights):<30} {prev_threshold:<12.2f} {prev_f1:<10.4f}")
print(f"{'NEW (optimized)':<30} {str((best_xgb_w, best_lgb_w, best_cat_w)):<30} {best_threshold:<12.2f} {best_config['f1']:<10.4f}")

improvement = best_config['f1'] - prev_f1
print(f"\nImprovement: {improvement:+.4f} ({improvement/prev_f1*100:+.2f}%)\n")

# ============================================================================
# STEP 6: VISUALIZATIONS
# ============================================================================
print("[STEP 6] Creating visualizations...\n")

# Plot 1: F1 vs Threshold for best weight combination
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter results for best weights
best_weight_results = results_df[
    (results_df['xgb_weight'] == best_xgb_w) &
    (results_df['lgb_weight'] == best_lgb_w) &
    (results_df['cat_weight'] == best_cat_w)
].sort_values('threshold')

ax = axes[0]
ax.plot(best_weight_results['threshold'], best_weight_results['f1'],
        marker='o', linewidth=2, markersize=6, color='#e74c3c')
ax.axvline(x=best_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal: {best_threshold:.2f}')
ax.set_xlabel('Threshold', fontsize=11, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=11, fontweight='bold')
ax.set_title(f'F1 vs Threshold\n(Weights: XGB {best_xgb_w:.2f}, LGB {best_lgb_w:.2f}, CAT {best_cat_w:.2f})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Weight combinations (top 20)
ax = axes[1]
top_20 = results_df.nlargest(20, 'f1')[['xgb_weight', 'lgb_weight', 'cat_weight', 'f1']].drop_duplicates()
weight_labels = [f"XGB {x:.2f}\nLGB {y:.2f}\nCAT {z:.2f}"
                 for x, y, z in zip(top_20['xgb_weight'], top_20['lgb_weight'], top_20['cat_weight'])]
colors_list = ['#e74c3c' if (x == best_xgb_w and y == best_lgb_w) else '#3498db'
               for x, y in zip(top_20['xgb_weight'], top_20['lgb_weight'])]

ax.barh(range(len(top_20)), top_20['f1'].values, color=colors_list)
ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(weight_labels, fontsize=8)
ax.set_xlabel('F1 Score', fontsize=11, fontweight='bold')
ax.set_title('Top 20 Weight Combinations', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('weight_threshold_optimization.png', dpi=150, bbox_inches='tight')
print("✓ Saved: weight_threshold_optimization.png\n")
plt.close()

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("=" * 80)
print("OPTIMIZATION COMPLETE")
print("=" * 80)

print(f"\n🎯 OPTIMAL CONFIGURATION FOUND:")
print(f"\n  XGBoost weight:  {best_xgb_w:.2f} ({best_xgb_w*100:.0f}%)")
print(f"  LightGBM weight: {best_lgb_w:.2f} ({best_lgb_w*100:.0f}%)")
print(f"  CatBoost weight: {best_cat_w:.2f} ({best_cat_w*100:.0f}%)")
print(f"  Threshold:       {best_threshold:.2f}")
print(f"\n  Validation F1:   {best_config['f1']:.4f}")
print(f"  Improvement:     {improvement:+.4f} vs previous")

print(f"\n📊 SUBMISSION FILE:")
print(f"  {filename}")
print(f"  Churn predictions: {churn_count:,} / {len(test_ids):,}")

print(f"\n💡 NEXT STEPS:")
print(f"  1. Submit: {filename}")
print(f"  2. If score is similar, try stacking or pseudo-labeling")
print(f"  3. Otherwise, fine-tune threshold further around {best_threshold:.2f}")

print("\n" + "=" * 80)


ENSEMBLE WEIGHT & THRESHOLD OPTIMIZATION

[STEP 1] Loading validation predictions...

✓ Validation set size: 118839
✓ Churn rate: 22.52%

✓ Test set size: 254655

[STEP 2] Grid search for optimal weights & threshold...

This may take a minute or two...

Testing 49 weight combinations × 27 thresholds = 1,323 total configs

  Progress: 4/49 weight configs tested...
  Progress: 8/49 weight configs tested...
  Progress: 12/49 weight configs tested...
  Progress: 16/49 weight configs tested...
  Progress: 20/49 weight configs tested...
  Progress: 24/49 weight configs tested...
  Progress: 28/49 weight configs tested...
  Progress: 32/49 weight configs tested...
  Progress: 36/49 weight configs tested...
  Progress: 40/49 weight configs tested...
  Progress: 44/49 weight configs tested...
  Progress: 48/49 weight configs tested...

✓ Grid search complete!

[STEP 3] Analyzing results...

TOP 10 CONFIGURATIONS:
----------------------------------------------------------------------------------

In [ ]:
"""
FAST WEIGHT & THRESHOLD OPTIMIZATION
===================================

Optimized version - calculates AUC once per weight combo (not per threshold)
Much faster: 3-5 mins instead of 1+ hour

Key optimization: AUC is threshold-independent, so only calculate once!
"""

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("=" * 80)
print("FAST WEIGHT & THRESHOLD OPTIMIZATION")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD VALIDATION PREDICTIONS
# ============================================================================
print("\n[STEP 1] Loading validation predictions...\n")

xgb_val_df = pd.read_csv('xgboost_val_predictions.csv')
lgb_val_df = pd.read_csv('lightgbm_val_predictions.csv')
cat_val_df = pd.read_csv('catboost_val_predictions.csv')

y_val_true = xgb_val_df['actual'].values
xgb_val_proba = xgb_val_df['predicted_proba'].values
lgb_val_proba = lgb_val_df['predicted_proba'].values
cat_val_proba = cat_val_df['predicted_proba'].values

print(f"✓ Validation set size: {len(y_val_true)}")
print(f"✓ Churn rate: {y_val_true.mean()*100:.2f}%\n")

# Load test probabilities
xgb_test_proba = pd.read_csv('xgboost_test_probabilities.csv')['churn_probability'].values
lgb_test_proba = pd.read_csv('lightgbm_test_probabilities.csv')['churn_probability'].values
cat_test_proba = pd.read_csv('catboost_test_probabilities.csv')['churn_probability'].values
test_ids = pd.read_csv('xgboost_submission.csv')['id'].values

print(f"✓ Test set size: {len(test_ids)}\n")

# ============================================================================
# STEP 2: FAST GRID SEARCH (OPTIMIZED)
# ============================================================================
print("[STEP 2] Fast grid search with optimized AUC calculation...\n")

# Define weight ranges with 0.2% (0.002) steps
weight_options = np.arange(200, 501, 2) / 1000  # 0.200, 0.202, ..., 0.500

# Thresholds: 0.34 to 0.40 with 0.002 steps
thresholds = np.arange(340, 401, 2) / 1000  # 0.340, 0.342, ..., 0.400

results = []
best_f1 = 0
best_config = None

# Count total configs
total_weight_combos = 0
for xgb_w in weight_options:
    for lgb_w in weight_options:
        cat_w = 1.0 - xgb_w - lgb_w
        if cat_w < 0.20 or cat_w > 0.50:
            continue
        total_weight_combos += 1

total_threshold_tests = total_weight_combos * len(thresholds)

print(f"Testing {total_weight_combos:,} weight combinations × {len(thresholds)} thresholds")
print(f"= {total_threshold_tests:,} total configurations\n")
print("Starting optimization...\n")

start_time = time.time()
combo_count = 0

for xgb_w in weight_options:
    for lgb_w in weight_options:
        cat_w = 1.0 - xgb_w - lgb_w
        if cat_w < 0.20 or cat_w > 0.50:
            continue

        # Create ensemble with these weights
        ensemble_val = (xgb_val_proba * xgb_w +
                       lgb_val_proba * lgb_w +
                       cat_val_proba * cat_w)

        # OPTIMIZATION: Calculate AUC ONCE per weight combo (not per threshold!)
        auc = roc_auc_score(y_val_true, ensemble_val)

        # Test different thresholds on THIS weight combo
        for threshold in thresholds:
            pred = (ensemble_val > threshold).astype(int)
            f1 = f1_score(y_val_true, pred)

            results.append({
                'xgb_weight': round(xgb_w, 4),
                'lgb_weight': round(lgb_w, 4),
                'cat_weight': round(cat_w, 4),
                'threshold': round(threshold, 4),
                'f1': f1,
                'auc': auc  # Same AUC for all thresholds of this weight combo
            })

            if f1 > best_f1:
                best_f1 = f1
                best_config = {
                    'xgb_weight': round(xgb_w, 4),
                    'lgb_weight': round(lgb_w, 4),
                    'cat_weight': round(cat_w, 4),
                    'threshold': round(threshold, 4),
                    'f1': f1,
                    'auc': auc
                }

        combo_count += 1
        elapsed = time.time() - start_time
        if combo_count % max(1, total_weight_combos // 10) == 0:
            progress_pct = (combo_count / total_weight_combos) * 100
            est_total = elapsed / (combo_count / total_weight_combos)
            est_remaining = est_total - elapsed
            print(f"  {progress_pct:.0f}% complete - {elapsed:.1f}s elapsed, ~{est_remaining:.1f}s remaining")

elapsed = time.time() - start_time
print(f"\n✓ Grid search complete in {elapsed:.1f} seconds!\n")

# ============================================================================
# STEP 3: ANALYZE RESULTS
# ============================================================================
print("[STEP 3] Analyzing results...\n")

results_df = pd.DataFrame(results)
best_results = results_df.nlargest(10, 'f1')

print("TOP 10 CONFIGURATIONS:")
print("-" * 110)
print(f"{'XGB':<12} {'LGB':<12} {'CAT':<12} {'Threshold':<12} {'F1':<12} {'AUC':<12}")
print("-" * 110)

for idx, row in best_results.iterrows():
    marker = " ✓ BEST" if row['f1'] == best_f1 else ""
    print(f"{row['xgb_weight']:<12.4f} {row['lgb_weight']:<12.4f} {row['cat_weight']:<12.4f} "
          f"{row['threshold']:<12.4f} {row['f1']:<12.6f} {row['auc']:<12.6f}{marker}")

# ============================================================================
# STEP 4: APPLY BEST CONFIG TO TEST SET
# ============================================================================
print(f"\n[STEP 4] Applying best configuration to test set...\n")

best_xgb_w = best_config['xgb_weight']
best_lgb_w = best_config['lgb_weight']
best_cat_w = best_config['cat_weight']
best_threshold = best_config['threshold']

print(f"Best weights found:")
print(f"  XGBoost:  {best_xgb_w:.4f} ({best_xgb_w*100:.2f}%)")
print(f"  LightGBM: {best_lgb_w:.4f} ({best_lgb_w*100:.2f}%)")
print(f"  CatBoost: {best_cat_w:.4f} ({best_cat_w*100:.2f}%)")
print(f"  Threshold: {best_threshold:.4f}")
print(f"  Validation F1: {best_config['f1']:.6f}")
print(f"  Validation AUC: {best_config['auc']:.6f}\n")

# Create ensemble with best weights
ensemble_test = (xgb_test_proba * best_xgb_w +
                 lgb_test_proba * best_lgb_w +
                 cat_test_proba * best_cat_w)

# Apply best threshold
test_pred = (ensemble_test > best_threshold).astype(int)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'churn': test_pred
})

filename = f'ensemble_optimized_fast_xgb{best_xgb_w:.4f}_lgb{best_lgb_w:.4f}_cat{best_cat_w:.4f}_th{best_threshold:.4f}.csv'
submission.to_csv(filename, index=False)

churn_count = test_pred.sum()
print(f"✓ Saved: {filename}")
print(f"  Churn predictions: {churn_count:,} / {len(test_ids):,}")
print(f"  Churn rate: {(churn_count/len(test_ids))*100:.2f}%\n")

# ============================================================================
# STEP 5: VISUALIZATIONS
# ============================================================================
print("[STEP 5] Creating visualizations...\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: F1 vs Threshold for best weight combo
best_weight_results = results_df[
    (results_df['xgb_weight'] == best_xgb_w) &
    (results_df['lgb_weight'] == best_lgb_w) &
    (results_df['cat_weight'] == best_cat_w)
].sort_values('threshold')

ax = axes[0, 0]
ax.plot(best_weight_results['threshold'], best_weight_results['f1'],
        marker='o', linewidth=2, markersize=4, color='#e74c3c')
ax.axvline(x=best_threshold, color='red', linestyle='--', linewidth=2,
           label=f'Optimal: {best_threshold:.4f}')
ax.set_xlabel('Threshold', fontsize=11, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=11, fontweight='bold')
ax.set_title(f'F1 vs Threshold (Best Weights)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: F1 distribution across all configs
ax = axes[0, 1]
ax.hist(results_df['f1'], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(x=best_f1, color='red', linestyle='--', linewidth=2, label=f'Best: {best_f1:.6f}')
ax.set_xlabel('F1 Score', fontsize=11, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=11, fontweight='bold')
ax.set_title('F1 Score Distribution', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: F1 vs XGB Weight
ax = axes[1, 0]
xgb_results = results_df.groupby('xgb_weight')['f1'].max().reset_index()
ax.plot(xgb_results['xgb_weight'], xgb_results['f1'],
        marker='o', linewidth=2, markersize=3, color='#e74c3c')
ax.axvline(x=best_xgb_w, color='red', linestyle='--', linewidth=2,
           label=f'Optimal: {best_xgb_w:.4f}')
ax.set_xlabel('XGBoost Weight', fontsize=11, fontweight='bold')
ax.set_ylabel('Best F1 Score', fontsize=11, fontweight='bold')
ax.set_title('F1 vs XGBoost Weight', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: F1 vs LGB Weight
ax = axes[1, 1]
lgb_results = results_df.groupby('lgb_weight')['f1'].max().reset_index()
ax.plot(lgb_results['lgb_weight'], lgb_results['f1'],
        marker='o', linewidth=2, markersize=3, color='#2ecc71')
ax.axvline(x=best_lgb_w, color='red', linestyle='--', linewidth=2,
           label=f'Optimal: {best_lgb_w:.4f}')
ax.set_xlabel('LightGBM Weight', fontsize=11, fontweight='bold')
ax.set_ylabel('Best F1 Score', fontsize=11, fontweight='bold')
ax.set_title('F1 vs LightGBM Weight', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fast_weight_optimization.png', dpi=150, bbox_inches='tight')
print("✓ Saved: fast_weight_optimization.png\n")
plt.close()

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("=" * 80)
print("OPTIMIZATION COMPLETE (FAST VERSION)")
print("=" * 80)

print(f"\n🎯 OPTIMAL CONFIGURATION:")
print(f"\n  XGBoost weight:  {best_xgb_w:.4f} ({best_xgb_w*100:.2f}%)")
print(f"  LightGBM weight: {best_lgb_w:.4f} ({best_lgb_w*100:.2f}%)")
print(f"  CatBoost weight: {best_cat_w:.4f} ({best_cat_w*100:.2f}%)")
print(f"  Threshold:       {best_threshold:.4f}")
print(f"\n  Validation F1:   {best_config['f1']:.6f}")
print(f"  Time taken:      {elapsed:.1f} seconds")

print(f"\n📊 SUBMISSION FILE:")
print(f"  {filename}")
print(f"  Churn predictions: {churn_count:,} / {len(test_ids):,}")

print(f"\n💡 NEXT STEPS:")
print(f"  1. Submit this file to Kaggle")
print(f"  2. If score improves, great!")
print(f"  3. If minimal improvement, try stacking")

print("\n" + "=" * 80)

FAST WEIGHT & THRESHOLD OPTIMIZATION

[STEP 1] Loading validation predictions...

✓ Validation set size: 118839
✓ Churn rate: 22.52%

✓ Test set size: 254655

[STEP 2] Fast grid search with optimized AUC calculation...

Testing 16,447 weight combinations × 31 thresholds
= 509,857 total configurations

Starting optimization...

  10% complete - 565.2s elapsed, ~5089.6s remaining
  20% complete - 1120.8s elapsed, ~4485.5s remaining
  30% complete - 1680.9s elapsed, ~3924.6s remaining
  40% complete - 2236.5s elapsed, ~3357.1s remaining
  50% complete - 2789.2s elapsed, ~2791.6s remaining
  60% complete - 3343.4s elapsed, ~2231.3s remaining
  70% complete - 3900.1s elapsed, ~1673.9s remaining
  80% complete - 4459.8s elapsed, ~1117.3s remaining
  90% complete - 5021.0s elapsed, ~560.3s remaining
  100% complete - 5579.6s elapsed, ~2.4s remaining

✓ Grid search complete in 5581.7 seconds!

[STEP 3] Analyzing results...

TOP 10 CONFIGURATIONS:
-----------------------------------------------

In [ ]:
"""
FINAL SUBMISSION - LOCKED WEIGHTS
=================================

Uses best performing weights:
- XGBoost: 0.20 (20%)
- LightGBM: 0.45 (45%)
- CatBoost: 0.35 (35%)

Optimize threshold only
"""

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
import matplotlib.pyplot as plt

print("=" * 80)
print("FINAL SUBMISSION - LOCKED WEIGHTS OPTIMIZATION")
print("=" * 80)

# ============================================================================
# STEP 1: LOAD VALIDATION PREDICTIONS
# ============================================================================
print("\n[STEP 1] Loading validation predictions...\n")

xgb_val_df = pd.read_csv('xgboost_val_predictions.csv')
lgb_val_df = pd.read_csv('lightgbm_val_predictions.csv')
cat_val_df = pd.read_csv('catboost_val_predictions.csv')

y_val_true = xgb_val_df['actual'].values
xgb_val_proba = xgb_val_df['predicted_proba'].values
lgb_val_proba = lgb_val_df['predicted_proba'].values
cat_val_proba = cat_val_df['predicted_proba'].values

print(f"✓ Validation set size: {len(y_val_true):,}")
print(f"✓ Churn rate: {y_val_true.mean()*100:.2f}%\n")

# Load test probabilities
xgb_test_proba = pd.read_csv('xgboost_test_probabilities.csv')['churn_probability'].values
lgb_test_proba = pd.read_csv('lightgbm_test_probabilities.csv')['churn_probability'].values
cat_test_proba = pd.read_csv('catboost_test_probabilities.csv')['churn_probability'].values
test_ids = pd.read_csv('xgboost_submission.csv')['id'].values

print(f"✓ Test set size: {len(test_ids):,}\n")

# ============================================================================
# STEP 2: CREATE ENSEMBLE WITH LOCKED WEIGHTS
# ============================================================================
print("[STEP 2] Creating ensemble with locked weights...\n")

xgb_w = 0.20
lgb_w = 0.45
cat_w = 0.35

print(f"Locked weights:")
print(f"  XGBoost:  {xgb_w:.2f} ({xgb_w*100:.0f}%)")
print(f"  LightGBM: {lgb_w:.2f} ({lgb_w*100:.0f}%)")
print(f"  CatBoost: {cat_w:.2f} ({cat_w*100:.0f}%)")
print(f"  Total: {xgb_w + lgb_w + cat_w:.2f}\n")

# Validation ensemble
ensemble_val = (xgb_val_proba * xgb_w +
               lgb_val_proba * lgb_w +
               cat_val_proba * cat_w)

# Test ensemble
ensemble_test = (xgb_test_proba * xgb_w +
                lgb_test_proba * lgb_w +
                cat_test_proba * cat_w)

print(f"✓ Ensemble created\n")

# ============================================================================
# STEP 3: FIND OPTIMAL THRESHOLD
# ============================================================================
print("[STEP 3] Finding optimal threshold...\n")

results = []
best_f1 = 0
best_threshold = 0.5

# Test thresholds with fine granularity
for threshold in np.arange(0.30, 0.56, 0.001):
    pred = (ensemble_val > threshold).astype(int)
    f1 = f1_score(y_val_true, pred)
    precision = precision_score(y_val_true, pred)
    recall = recall_score(y_val_true, pred)
    auc = roc_auc_score(y_val_true, ensemble_val)

    results.append({
        'threshold': round(threshold, 3),
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc': auc
    })

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

results_df = pd.DataFrame(results)
top_results = results_df.nlargest(10, 'f1')

print("TOP 10 THRESHOLDS:")
print("-" * 80)
print(f"{'Threshold':<12} {'F1':<12} {'Precision':<12} {'Recall':<12} {'AUC':<12}")
print("-" * 80)

for idx, row in top_results.iterrows():
    marker = " ✓ BEST" if row['f1'] == best_f1 else ""
    print(f"{row['threshold']:<12.3f} {row['f1']:<12.6f} {row['precision']:<12.6f} "
          f"{row['recall']:<12.6f} {row['auc']:<12.6f}{marker}")

print(f"\n✓ Optimal threshold: {best_threshold:.3f}")
print(f"  Validation F1: {best_f1:.6f}\n")

# ============================================================================
# STEP 4: GENERATE FINAL SUBMISSION
# ============================================================================
print("[STEP 4] Generating final submission...\n")

test_pred = (ensemble_test > best_threshold).astype(int)

submission = pd.DataFrame({
    'id': test_ids,
    'churn': test_pred
})

filename = f'FINAL_submission_xgb0.20_lgb0.45_cat0.35_th{best_threshold:.3f}.csv'
submission.to_csv(filename, index=False)

churn_count = test_pred.sum()
churn_pct = (churn_count / len(test_ids)) * 100

print(f"✓ Saved: {filename}")
print(f"  Churn predictions: {churn_count:,} / {len(test_ids):,}")
print(f"  Churn rate: {churn_pct:.2f}%\n")

# ============================================================================
# STEP 5: VISUALIZATION
# ============================================================================
print("[STEP 5] Creating visualization...\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: F1, Precision, Recall vs Threshold
ax = axes[0]
ax.plot(results_df['threshold'], results_df['f1'], marker='o', linewidth=2,
        markersize=3, label='F1', color='#e74c3c')
ax.plot(results_df['threshold'], results_df['precision'], marker='s', linewidth=2,
        markersize=2, label='Precision', color='#3498db', alpha=0.7)
ax.plot(results_df['threshold'], results_df['recall'], marker='^', linewidth=2,
        markersize=2, label='Recall', color='#2ecc71', alpha=0.7)
ax.axvline(x=best_threshold, color='red', linestyle='--', linewidth=2,
           label=f'Optimal: {best_threshold:.3f}')
ax.set_xlabel('Threshold', fontsize=11, fontweight='bold')
ax.set_ylabel('Score', fontsize=11, fontweight='bold')
ax.set_title('Metrics vs Threshold', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.6, 0.75])

# Plot 2: F1 zoomed in around optimal
ax = axes[1]
zoom_range = (best_threshold - 0.02, best_threshold + 0.02)
zoom_results = results_df[(results_df['threshold'] >= zoom_range[0]) &
                          (results_df['threshold'] <= zoom_range[1])]
ax.plot(zoom_results['threshold'], zoom_results['f1'], marker='o', linewidth=2.5,
        markersize=5, color='#e74c3c')
ax.axvline(x=best_threshold, color='red', linestyle='--', linewidth=2,
           label=f'Optimal: {best_threshold:.3f}')
ax.axhline(y=best_f1, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Threshold', fontsize=11, fontweight='bold')
ax.set_ylabel('F1 Score', fontsize=11, fontweight='bold')
ax.set_title('F1 Score (Zoomed in)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('final_submission_optimization.png', dpi=150, bbox_inches='tight')
print("✓ Saved: final_submission_optimization.png\n")
plt.close()

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("=" * 80)
print("FINAL SUBMISSION READY")
print("=" * 80)

print(f"\n🎯 CONFIGURATION:")
print(f"   XGBoost weight:  0.20 (20%)")
print(f"   LightGBM weight: 0.45 (45%)")
print(f"   CatBoost weight: 0.35 (35%)")
print(f"   Threshold:       {best_threshold:.3f}")

print(f"\n📊 VALIDATION PERFORMANCE:")
print(f"   F1 Score:        {best_f1:.6f}")
print(f"   AUC-ROC:         {results_df[results_df['threshold'] == round(best_threshold, 3)]['auc'].values[0]:.6f}")

print(f"\n📁 SUBMISSION FILE:")
print(f"   {filename}")
print(f"   Size: {len(submission):,} rows")
print(f"   Churn predictions: {churn_count:,} ({churn_pct:.2f}%)")

print(f"\n✅ READY TO SUBMIT TO KAGGLE!")

print("\n" + "=" * 80)

FINAL SUBMISSION - LOCKED WEIGHTS OPTIMIZATION

[STEP 1] Loading validation predictions...

✓ Validation set size: 118,839
✓ Churn rate: 22.52%

✓ Test set size: 254,655

[STEP 2] Creating ensemble with locked weights...

Locked weights:
  XGBoost:  0.20 (20%)
  LightGBM: 0.45 (45%)
  CatBoost: 0.35 (35%)
  Total: 1.00

✓ Ensemble created

[STEP 3] Finding optimal threshold...

TOP 10 THRESHOLDS:
--------------------------------------------------------------------------------
Threshold    F1           Precision    Recall       AUC         
--------------------------------------------------------------------------------
0.370        0.708256     0.648370     0.780331     0.920213     ✓ BEST
0.371        0.708125     0.648900     0.779247     0.920213    
0.366        0.708106     0.646147     0.783208     0.920213    
0.369        0.708105     0.647730     0.780892     0.920213    
0.356        0.708095     0.640737     0.791279     0.920213    
0.363        0.708007     0.644214     0.